## SAX, Hierarchical clustering

This notebook has the following sections:

1. `tslearn` SAX representations,
2. SAX MINDIST distance on symbolic strings,
3. agglomerative hierarchical clustering with complete linkage,
4. subsample-based clustering stability by `k`,
5. explainable cluster-level Google Trends shape and words

Outputs are mostly CSV files and some diagnostic figures.

## 0. Configuration

Edit paths and headline modeling choices here. The rest of the notebook runs top-to-bottom.

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.signal import savgol_filter
from scipy.spatial.distance import squareform, pdist
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from tslearn.piecewise import SymbolicAggregateApproximation, PiecewiseAggregateApproximation

# -----------------------------------------------------------------------------
# PATHS -- environment-variable overridable
# -----------------------------------------------------------------------------
# relative defaults so the
# notebook runs out of the box for any collaborator who clones the repo.
# Override via a .env file or shell export, e.g.:
#   export SAX_DATA_DIR=/path/to/data_events
#   export SAX_OUTPUT_DIR=/path/to/output/run_01
# DATA_DIR = Path(os.environ.get("SAX_DATA_DIR", "./data/data_events"))
# OUTPUT_DIR = Path(os.environ.get("SAX_OUTPUT_DIR", "./output/sax_run"))

# SP500_PATH = Path(os.environ.get("SAX_SP500_PATH", "./data/shock_detection/SP500_data.csv"))
# DJ_PATH = Path(os.environ.get("SAX_DJ_PATH", "./data/shock_detection/DOWJONES_data.csv"))
# NASDAQ_PATH = Path(os.environ.get("SAX_NASDAQ_PATH", "./data/shock_detection/NASDAQ100_data.csv"))
# RUSSELL_PATH = Path(os.environ.get("SAX_RUSSELL_PATH", "./data/shock_detection/RUSSELL2000_data.csv"))

DATA_DIR = Path(r"C:\Python\CSUREMM\data\processed_gt")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output")

SP500_PATH = Path(r"C:\Python\CSUREMM\data\shock_detection\SP500_data.csv")
DJ_PATH = Path(r"C:\Python\CSUREMM\data\shock_detection\DOWJONES_data.csv")
NASDAQ_PATH = Path(r"C:\Python\CSUREMM\data\shock_detection\NASDAQ100_data.csv")
RUSSELL_PATH = Path(r"C:\Python\CSUREMM\data\shock_detection\RUSSELL2000_data.csv")

STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed02*.csv"

START_DATE = "2022-01-01"
END_DATE = "2026-05-31"

# -----------------------------------------------------------------------------
# Data-specific filtering and preprocessing
# -----------------------------------------------------------------------------
MAX_ZERO_SHARE = 0.50
MAX_MISSING_SHARE = 0.02
MAX_INTERPOLATE_GAP = 7

DENOISE_WINDOW = 9
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91

# -----------------------------------------------------------------------------
# SAX and hierarchical clustering
# -----------------------------------------------------------------------------
MINDIST_CHUNK_SIZE = 256
K_RANGE = range(2, 13)
RANDOM_STATE = 42
N_SEGMENTS = 96
ALPHABET_SIZE = 9
FINAL_K = 10                       # primary k: see candidate_k_interpretation_summary.csv
CANDIDATE_K_VALUES = [7, 8, 9, 10, 11]  # k values fit and compared side-by-side in Steps 4-5

if FINAL_K not in CANDIDATE_K_VALUES:
    raise ValueError(
        f"FINAL_K={FINAL_K} must be one of CANDIDATE_K_VALUES={CANDIDATE_K_VALUES}. "
        "Add it to the list, or change FINAL_K to a value already being compared."
    )

# PAA BREAKPOINT_MODE = "empirical" derives breakpoints from the observed distribution
# instead of theoretical N(0,1) quantiles. Set back to "gaussian" to reproduce
# the original SAX behavior.
BREAKPOINT_MODE = "empirical"       # {"gaussian", "empirical"}

# This grid is for a diagnostic that reports how tight MINDIST is against true Euclidean distance
# at each (w, a) combination to justify the parameter selection
SEGMENT_GRID = [48, 64, 96, 128]
ALPHABET_GRID = [5, 7, 9, 11]
ABLATION_SAMPLE_SIZE = 150          # terms subsampled for the (w, a) tightness check

# Default to ward linkage, but empirically compare all four candidates
LINKAGE_METHOD = "ward"           # {"single", "average", "complete", "ward"}
LINKAGE_CANDIDATES = ["single", "average", "complete", "ward"]

# MINDIST produces exact/near-exact ties for genuinely distinct series
# (a direct consequence of quantization). Within TIE_EPS of each other, merge
# order is broken using the true Euclidean distance on the underlying series
# instead of being left to array/index order inside scipy's linkage.
TIE_EPS = 1e-6

# Step 5 also logs the stability/silhouette-recommended k (subject to this
# floor) alongside FINAL_K for comparison; it never overrides FINAL_K.
MIN_ACCEPTABLE_CLUSTER_SIZE = 15

# Stability: each iteration draws two overlapping subsamples and compares labels
STABILITY_SUBSAMPLES = 80
STABILITY_FRACTION = 0.80

# Cluster-index construction (used in Section 5)
CORE_STABILITY_QUANTILE = 0.75
MIN_CORE_TERMS = 10
TOP_N_CORE_TERMS = 10

SUBDIRS = [
    "00_provenance",
    "01_preprocessing",
    "02_sax",
    "03_distance",
    "04_clustering",
    "05_validation",
    "06_robustness",
    "07_visualization",
]

for sub in SUBDIRS:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Python\CSUREMM\output\sax_tests_july_14


## 1. Load, filter, denoise, detrend, and normalize

In [2]:
def clean_term_from_folder(folder: Path) -> str:
    return folder.name.replace("_", " ").strip()


def find_value_column(df: pd.DataFrame) -> str:
    date_like = {"time", "date", "week"}
    candidates = [c for c in df.columns if c.lower() not in date_like]

    if not candidates:
        raise ValueError("No value column found.")

    numeric = [
        c for c in candidates
        if pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors="coerce"))
    ]

    return numeric[0] if numeric else candidates[0]


def max_consecutive_missing(s: pd.Series) -> int:
    missing = s.isna()

    if not missing.any():
        return 0

    groups = (missing != missing.shift()).cumsum()

    return int(
        missing
        .groupby(groups)
        .sum()
        .max()
    )


def load_one_series(folder: Path) -> pd.Series:
    stitched_dir = folder / STITCHED_SUBDIR
    files = sorted(stitched_dir.glob(STITCHED_GLOB)) if stitched_dir.exists() else []

    if not files:
        raise FileNotFoundError("missing stitched file")

    path = files[-1]
    df = pd.read_csv(path)

    date_col = next(
        (c for c in df.columns if c.lower() in {"time", "date", "week"}),
        df.columns[0],
    )

    value_col = find_value_column(df)

    dates = pd.to_datetime(df[date_col], errors="coerce")
    values = pd.to_numeric(df[value_col], errors="coerce")

    s = pd.Series(
        values.values,
        index=dates,
        name=clean_term_from_folder(folder),
    )

    s = s[~s.index.isna()]
    s = s[~s.index.duplicated(keep="last")]
    s = s.sort_index()

    s = s.loc[START_DATE:END_DATE]

    full_index = pd.date_range(
        START_DATE,
        END_DATE,
        freq="D",
    )

    return s.reindex(full_index)


def build_panel(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    series = {}

    folders = sorted([p for p in data_dir.iterdir() if p.is_dir()])

    expected_days = len(
        pd.date_range(
            START_DATE,
            END_DATE,
            freq="D",
        )
    )

    for folder in folders:
        term = clean_term_from_folder(folder)

        try:
            s = load_one_series(folder)

            observed_share = float(s.notna().mean())
            missing_share = float(s.isna().mean())
            zero_share = float((s.dropna() == 0).mean())
            longest_missing_gap = max_consecutive_missing(s)
            n_unique = int(s.nunique(dropna=True))

            reason = "kept"

            if zero_share > MAX_ZERO_SHARE:
                reason = "high_zero_share"
            elif missing_share > MAX_MISSING_SHARE:
                reason = "high_missing_share"
            elif longest_missing_gap > MAX_INTERPOLATE_GAP:
                reason = "long_missing_gap"
            elif n_unique <= 2:
                reason = "near_constant"

            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": int(s.notna().sum()),
                "observed_share": observed_share,
                "missing_share": missing_share,
                "zero_share": zero_share,
                "longest_missing_gap": longest_missing_gap,
                "n_unique": n_unique,
                "drop_reason": reason,
            })

            if reason == "kept":
                series[term] = s

        except Exception as e:
            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": 0,
                "observed_share": 0.0,
                "missing_share": 1.0,
                "zero_share": np.nan,
                "longest_missing_gap": np.nan,
                "n_unique": 0,
                "drop_reason": str(e),
            })

    panel = pd.DataFrame(series).sort_index()
    funnel = pd.DataFrame(rows).sort_values(["drop_reason", "term"])

    return panel, funnel


def interpolate_small_gaps(s: pd.Series) -> pd.Series:
    return (
        s.astype(float)
        .interpolate(
            method="time",
            limit=MAX_INTERPOLATE_GAP,
            limit_direction="both",
        )
    )


def denoise_series(s: pd.Series) -> pd.Series:
    x = s.astype(float)
    valid = x.dropna()

    if len(valid) < DENOISE_WINDOW or DENOISE_WINDOW % 2 == 0:
        return x

    filled = x.interpolate(
        method="time",
        limit_direction="both",
    )

    return pd.Series(
        savgol_filter(
            filled,
            DENOISE_WINDOW,
            DENOISE_POLYORDER,
        ),
        index=x.index,
        name=x.name,
    )


def detrend_series(s: pd.Series) -> pd.Series:
    trend = s.rolling(
        DETREND_WINDOW,
        center=True,
        min_periods=max(14, DETREND_WINDOW // 4),
    ).median()

    return s - trend


def robust_zscore(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    x = s.astype(float)

    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)

        if pd.isna(std) or std == 0:
            return pd.Series(
                np.zeros(len(x)),
                index=x.index,
                name=x.name,
            )

        return (
            (x - x.mean(skipna=True)) / std
        ).rename(x.name)

    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    mad_ratio = (
        mad / wide_spread
        if wide_spread > 0
        else 1.0
    )

    is_degenerate = mad_ratio < mad_ratio_threshold

    if not is_degenerate:
        return (
            0.6745 * (x - med) / mad
        ).rename(x.name)

    mad_floored = (
        max(mad, mad_floor_frac * wide_spread)
        if wide_spread > 0
        else mad
    )

    z = (
        0.6745 * (x - med) / mad_floored
    ).rename(x.name)

    return z.clip(
        lower=-clip_mad_multiple,
        upper=clip_mad_multiple,
    )


def preprocess_panel(
    panel: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:

    stages = {}

    stages["filled"] = panel.apply(
        interpolate_small_gaps,
        axis=0,
    )

    stages["denoised"] = stages["filled"].apply(
        denoise_series,
        axis=0,
    )

    stages["detrended"] = stages["denoised"].apply(
        detrend_series,
        axis=0,
    )

    stages["normalized"] = stages["detrended"].apply(
        robust_zscore,
        axis=0,
    )

    return stages["normalized"], stages


def save_panel_stages(panel_raw: pd.DataFrame, stages: dict[str, pd.DataFrame]) -> None:
    panel_raw.to_csv(OUTPUT_DIR / "01_preprocessing" / "panel_raw_retained.csv")
    for name, df in stages.items():
        df.to_csv(OUTPUT_DIR / "01_preprocessing" / f"panel_{name}.csv")

In [3]:
panel_raw, filtering_funnel = build_panel(DATA_DIR)
filtering_funnel.to_csv(OUTPUT_DIR / "00_provenance" / "filtering_funnel.csv", index=False)

panel_norm, stages = preprocess_panel(panel_raw)
panel_norm = panel_norm.dropna(axis=1, how="all")
panel_norm = panel_norm.loc[:, panel_norm.notna().mean() > 0.95]
stages["normalized"] = panel_norm

save_panel_stages(panel_raw, stages)

summary = pd.DataFrame({
    "n_days": [panel_norm.shape[0]],
    "n_terms_retained": [panel_norm.shape[1]],
    "start_date": [panel_norm.index.min()],
    "end_date": [panel_norm.index.max()],
})
summary.to_csv(OUTPUT_DIR / "00_provenance" / "analysis_sample_summary.csv", index=False)

print(summary.to_string(index=False))
print("\nFiltering funnel:")
print(filtering_funnel["drop_reason"].value_counts(dropna=False).to_string())

 n_days  n_terms_retained start_date   end_date
   1612               847 2022-01-01 2026-05-31

Filtering funnel:
drop_reason
kept                  847
high_zero_share       352
high_missing_share      4


In [4]:
panel_norm.isna().sum().sum() == 0

np.True_

## 2. `tslearn` SAX representation

Each retained term is represented as one SAX string. The notebook uses `tslearn.piecewise.SymbolicAggregateApproximation`. Linearly fills missing values, only for the representation step, though previous steps should have already produced a clean panel without NAs.

In [5]:
def panel_to_tslearn_array(panel: pd.DataFrame) -> np.ndarray:
    filled = panel.astype(float).interpolate("time").ffill().bfill()
    X = filled.T.to_numpy(dtype=float)
    return X[:, :, None]


def sax_to_strings(codes_2d: np.ndarray, alphabet_size: int, index) -> pd.Series:
    alphabet = np.array(list("abcdefghijklmnopqrstuvwxyz"[:alphabet_size]))
    strings = ["".join(alphabet[row]) for row in codes_2d]
    return pd.Series(strings, index=index, name="sax_string")


def sax_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian breakpoints used by standard SAX (theoretical N(0,1) assumption)."""
    return norm.ppf(np.arange(1, alphabet_size) / alphabet_size)


def empirical_breakpoints(paa_values: np.ndarray, alphabet_size: int) -> np.ndarray:
    """
    Data-driven analogue of sax_breakpoints(): equiprobable quantiles of the
    pooled, observed PAA-coefficient distribution rather than theoretical
    N(0,1) quantiles. Use when the Gaussian assumption doesn't fit the data
    (see the fit diagnostic immediately below).
    """
    qs = np.arange(1, alphabet_size) / alphabet_size
    return np.quantile(paa_values, qs)


def discretize_paa(paa_values_2d: np.ndarray, breakpoints: np.ndarray) -> np.ndarray:
    """Assign each PAA coefficient to a symbol index given a set of breakpoints."""
    return np.digitize(paa_values_2d, breakpoints)


# throwing away extreme values to avoid distortion in SAX representation
panel_sax = panel_norm.clip(-5, 5)
X_ts = panel_to_tslearn_array(panel_sax)

paa = PiecewiseAggregateApproximation(n_segments=N_SEGMENTS)
paa_coefs = paa.fit_transform(X_ts)[:, :, 0]          # (n_terms, n_segments)
paa_flat = paa_coefs.ravel()

# BREAKPOINT_MODE (config, Section 0) selects Gaussian vs. empirical breakpoints.
if BREAKPOINT_MODE == "empirical":
    active_breakpoints = empirical_breakpoints(paa_flat, ALPHABET_SIZE)
else:
    active_breakpoints = sax_breakpoints(ALPHABET_SIZE)

sax_code_array = discretize_paa(paa_coefs, active_breakpoints)   # (n_terms, n_segments)
sax_codes = sax_code_array[:, :, None]                            # tslearn-style shape

sax_strings = sax_to_strings(sax_code_array, ALPHABET_SIZE, panel_norm.columns)

sax_df = pd.DataFrame({
    "term": sax_strings.index,
    "sax_string": sax_strings.values,
    "n_segments": N_SEGMENTS,
    "alphabet_size": ALPHABET_SIZE,
    "breakpoint_mode": BREAKPOINT_MODE,
})
sax_df.to_csv(OUTPUT_DIR / "02_sax" / "sax_strings.csv", index=False)

symbol_counts = (
    sax_df.assign(symbol=sax_df["sax_string"].map(list))
    .explode("symbol")
    .groupby(["term", "symbol"])
    .size()
    .unstack(fill_value=0)
)
symbol_counts.to_csv(OUTPUT_DIR / "02_sax" / "sax_symbol_counts.csv")

print("SAX strings:", sax_df.shape, " | breakpoint_mode:", BREAKPOINT_MODE)
sax_df.head()

SAX strings: (847, 5)  | breakpoint_mode: empirical


,term,sax_string,n_segments,alphabet_size,breakpoint_mode
0,2020 election map,gfadaebheagifafbebhicfbbaggbggcbfcbaiiaedbiadb...,96,9,empirical
1,2020 election results,fgbgfcchieihahhcaagihhcbeefehdedecbbiidaahiahf...,96,9,empirical
2,2022 election,bcgihaaiifbaaiiceadigeeeeeeeedgchfbdhdbcghidca...,96,9,empirical
3,2022 election results,edehgfchihgcaiieecdigeeeeeeeedfdfebfgecdfeibbb...,96,9,empirical
4,2024 election,gdcfcbeedahhhbccdbfihbabghiahgggheaagigaaiiaaa...,96,9,empirical


### Justify clipping value selection at $\pm 5 \text{ SD}$ is the right approach

In [6]:
for c in [2,3,4,5]:
    pct = np.mean(np.abs(panel_norm.to_numpy()) > c)
    print(c, pct)

2 0.18607492214530338
3 0.1243272856176082
4 0.09299351674718244
5 0.07572705886488877


In [7]:
# --- Diagnostic: does the Gaussian breakpoint assumption actually fit? ---
# Excess kurtosis > 0 or a small KS p-value indicates the pooled coefficients are more peaked / non-Gaussian
# which motivates BREAKPOINT_MODE = "empirical" above.
from scipy import stats as _stats

standardized = (paa_flat - paa_flat.mean()) / paa_flat.std()
excess_kurtosis = _stats.kurtosis(paa_flat, fisher=True)   # 0 for a true Gaussian
ks_stat, ks_p = _stats.kstest(standardized, "norm")

breakpoint_fit = pd.DataFrame({
    "metric": ["excess_kurtosis", "ks_statistic", "ks_pvalue", "n_paa_coefficients", "breakpoint_mode_used"],
    "value": [excess_kurtosis, ks_stat, ks_p, len(paa_flat), BREAKPOINT_MODE],
})
breakpoint_fit.to_csv(OUTPUT_DIR / "06_robustness" / "paa_gaussian_fit_diagnostic.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(paa_flat, bins=60, density=True, alpha=0.7, label="pooled PAA coefficients")
xs = np.linspace(paa_flat.min(), paa_flat.max(), 200)
axes[0].plot(xs, norm.pdf(xs, paa_flat.mean(), paa_flat.std()), lw=2, label="fitted Gaussian")
axes[0].set_title("Pooled PAA coefficients vs Gaussian fit")
axes[0].legend(frameon=False)
_stats.probplot(paa_flat, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot vs Gaussian")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_robustness" / "paa_gaussian_fit_diagnostic.png", dpi=200)
plt.close(fig)

print(f"Excess kurtosis: {excess_kurtosis:.3f}  (0 = Gaussian; > 0 = more peaked than Gaussian)")
print(f"KS test vs N(0,1): stat={ks_stat:.4f}, p={ks_p:.2e}")
print(f"-> breakpoint_mode currently in use: {BREAKPOINT_MODE}")

Excess kurtosis: 3.887  (0 = Gaussian; > 0 = more peaked than Gaussian)
KS test vs N(0,1): stat=0.1457, p=0.00e+00
-> breakpoint_mode currently in use: empirical


## 3. MINDIST

The clustering distance is SAX **MINDIST**, computed from the (Gaussian or
empirical, per `BREAKPOINT_MODE`) breakpoints from Section 2.

Before committing to `N_SEGMENTS` / `ALPHABET_SIZE`, a small ablation below
reports how tight MINDIST is against the true Euclidean distance across a grid
of `(n_segments, alphabet_size)` choices -- this is a diagnostic, not an
automatic override, so the hardcoded values above can be checked against
evidence.

Linkage defaults to **ward linkage**: it is the only merge rule for which
Kleinberg-style scale invariance + richness + consistency are jointly
satisfiable once stated over the dendrogram (Zadeh & Ben-David, 2009), which
is the property MINDIST-based clustering is implicitly leaning on. All four
standard linkage methods are compared empirically in Section 4 regardless, so
this default is a starting point, not an assumption baked in silently.

MINDIST also produces exact/near-exact distance ties for genuinely distinct
series (a direct consequence of quantization); these are broken using the true
Euclidean distance on the underlying series rather than left to array order.

In [8]:
def sax_symbol_distance_table(alphabet_size: int, breakpoints: np.ndarray | None = None) -> np.ndarray:
    """
    Pairwise SAX symbol distances used in MINDIST.

    Adjacent symbols have distance 0 because their intervals touch. Non-adjacent
    symbols are separated by the gap between the relevant breakpoints. Accepts an
    explicit breakpoints array so the table reflects whichever BREAKPOINT_MODE
    (Gaussian or empirical) was actually used to build the SAX codes.
    """
    bp = sax_breakpoints(alphabet_size) if breakpoints is None else breakpoints
    table = np.zeros((alphabet_size, alphabet_size), dtype=float)

    for i in range(alphabet_size):
        for j in range(alphabet_size):
            if abs(i - j) <= 1:
                table[i, j] = 0.0
            else:
                lo, hi = sorted((i, j))
                table[i, j] = bp[hi - 1] - bp[lo]
    return table

# NOTE: the (w, a) MINDIST-tightness ablation that used to run here has moved
# to Section 06 (Validity checks), alongside the reconstruction-elbow and
# downstream-quality checks it's meant to be read together with.

In [9]:
def sax_mindist_matrix(
    sax_codes: np.ndarray,
    terms: list[str],
    n_timestamps: int,
    n_segments: int,
    alphabet_size: int,
    symbol_dist: np.ndarray,
    chunk_size: int = MINDIST_CHUNK_SIZE,
) -> pd.DataFrame:
    """
    Compute SAX MINDIST between all term-level SAX strings.

    Parameters
    ----------
    sax_codes:
        Symbol-code array, shape (n_terms, n_segments, 1).
    terms:
        Term names in the same order as sax_codes.
    n_timestamps:
        Original equal-length series length before SAX compression.
    n_segments:
        Number of SAX/PAA segments.
    alphabet_size:
        Number of SAX symbols.
    symbol_dist:
        Precomputed symbol-to-symbol distance table (reflects the breakpoints
        actually used -- Gaussian or empirical -- rather than assuming Gaussian).
    chunk_size:
        Row chunk size to keep memory use stable.

    Returns
    -------
    pd.DataFrame
        Symmetric MINDIST matrix indexed and columned by term.
    """
    codes = np.asarray(sax_codes[:, :, 0], dtype=np.int16)
    n_terms, actual_segments = codes.shape

    if actual_segments != n_segments:
        raise ValueError(f"Expected {n_segments} SAX segments, got {actual_segments}.")
    if len(terms) != n_terms:
        raise ValueError("Number of terms does not match number of SAX code rows.")

    scale = np.sqrt(n_timestamps / n_segments)
    D = np.zeros((n_terms, n_terms), dtype=np.float32)

    for start in range(0, n_terms, chunk_size):
        stop = min(start + chunk_size, n_terms)
        block = codes[start:stop]
        per_segment = symbol_dist[block[:, None, :], codes[None, :, :]]
        D[start:stop, :] = scale * np.sqrt((per_segment ** 2).sum(axis=2))

    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=terms, columns=terms)


def true_euclidean_distance_matrix(panel: pd.DataFrame) -> pd.DataFrame:
    """True (full-resolution) Euclidean distance, used only to break MINDIST ties."""
    X = panel.T.to_numpy(dtype=float)
    D = squareform(pdist(X, metric="euclidean"))
    return pd.DataFrame(D, index=panel.columns, columns=panel.columns)


def break_mindist_ties(mindist_df: pd.DataFrame, euclid_df: pd.DataFrame, eps: float = TIE_EPS) -> pd.DataFrame:
    """
    MINDIST is a many-to-one lower bound and produces exact/near-exact ties for
    genuinely distinct series (see the duplicate-value counts in the SAX
    characterization diagnostics). Within `eps` those ties are effectively
    arbitrary and would otherwise be resolved by array/index order inside
    scipy's linkage. This adds a rank-preserving nudge from the true Euclidean
    distance, strictly smaller than `eps`, so no non-tied comparison is
    reordered but tied comparisons are resolved using real signal.
    """
    M = mindist_df.to_numpy(dtype=float)
    E = euclid_df.reindex(index=mindist_df.index, columns=mindist_df.columns).to_numpy(dtype=float)

    e_rank = E / (E.max() + 1e-9)          # in [0, 1), preserves Euclidean order
    nudge = eps * e_rank                    # strictly smaller than eps
    M_broken = M + nudge
    np.fill_diagonal(M_broken, 0.0)
    M_broken = 0.5 * (M_broken + M_broken.T)
    return pd.DataFrame(M_broken, index=mindist_df.index, columns=mindist_df.columns)


def hierarchical_labels(distance_df: pd.DataFrame, k: int, method: str | None = None) -> pd.Series:
    method = method or LINKAGE_METHOD
    condensed = squareform(distance_df.to_numpy(dtype=float), checks=False)
    Z = linkage(condensed, method=method)
    labels = fcluster(Z, t=k, criterion="maxclust")
    return pd.Series(labels, index=distance_df.index, name="cluster")


symbol_dist_table = sax_symbol_distance_table(ALPHABET_SIZE, active_breakpoints)

mindist = sax_mindist_matrix(
    sax_codes=sax_codes,
    terms=sax_strings.index.to_list(),
    n_timestamps=panel_norm.shape[0],
    n_segments=N_SEGMENTS,
    alphabet_size=ALPHABET_SIZE,
    symbol_dist=symbol_dist_table,
)
mindist.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_matrix.csv")

mindist_summary = pd.DataFrame({
    "metric": ["n_terms", "n_segments", "alphabet_size", "min_offdiag", "median_offdiag", "mean_offdiag", "max_offdiag"],
    "value": [
        mindist.shape[0],
        N_SEGMENTS,
        ALPHABET_SIZE,
        np.nanmin(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmedian(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmean(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmax(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
    ],
})
mindist_summary.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_summary.csv", index=False)

n_exact_ties = int(np.isclose(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy(), 0.0, atol=1e-9).sum())
print(f"Exact-zero off-diagonal MINDIST pairs before tie-breaking: {n_exact_ties}")

euclid_true = true_euclidean_distance_matrix(panel_sax)
mindist_tiebroken = break_mindist_ties(mindist, euclid_true, eps=TIE_EPS)
mindist_tiebroken.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_matrix_tiebroken.csv")

mindist_summary

Exact-zero off-diagonal MINDIST pairs before tie-breaking: 16


,metric,value
0,n_terms,847.000000
1,n_segments,96.000000
2,alphabet_size,9.000000
3,min_offdiag,0.000000
4,median_offdiag,32.555050
5,mean_offdiag,31.949610
6,max_offdiag,54.985077


### Some diagnosis on SAX characterization

In [10]:
print("panel_norm shape:", panel_norm.shape)
print("N_SEGMENTS:", N_SEGMENTS)
print("ALPHABET_SIZE:", ALPHABET_SIZE)
print("FINAL_K:", FINAL_K)
print("NaNs:", panel_norm.isna().sum().sum())
print("Constant cols:", (panel_norm.std(axis=0) == 0).sum())

panel_norm shape: (1612, 847)
N_SEGMENTS: 96
ALPHABET_SIZE: 9
FINAL_K: 10
NaNs: 0
Constant cols: 0


In [11]:
codes = np.asarray(sax_codes[:, :, 0], dtype=int)

# 1. How many unique SAX words?
sax_words = ["".join(map(str, row)) for row in codes]
print("unique SAX words:", pd.Series(sax_words).nunique())
print(pd.Series(sax_words).value_counts().head(20))

# 2. Symbol usage
symbol_counts = pd.Series(codes.ravel()).value_counts().sort_index()
print(symbol_counts)

# 3. Per-term number of unique symbols
unique_symbols_per_term = pd.Series(
    [len(set(row)) for row in codes],
    index=sax_strings.index,
    name="n_unique_symbols"
)
print(unique_symbols_per_term.describe())
print(unique_symbols_per_term.value_counts().sort_index())

# 4. Distance distribution
offdiag = mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy().ravel()
offdiag = offdiag[~np.isnan(offdiag)]
print(pd.Series(offdiag).describe())
print(pd.Series(np.round(offdiag, 6)).value_counts().head(20))

unique SAX words: 843
434765278762088442386444444443535415642354811188478000377058100188803575828583466874443885008812    2
632521443077712231587101678076667400686008800080388000660588080367444444443434633752236744215743    2
007810088210088450187644445453687413724174871065786422671777170367444444444773573721133867203748    2
136206577531215576663704766516773321165766617047465167652112676457150565643575512145666562103746    2
650304174068505141782511066166215210880431803187087200430788171367444444444444444645545786106851    1
561652278487077200687721445473434211883007807577186211371688080178444444444543644881025772808880    1
126870088510088240386444444443627513731267832044376331141477011688410444424162266880147786008850    1
414621352177723140387031588066777701686008800186088101170688080367444444443434634862245743116861    1
047208788641226786770701777316884312277767706008662276552223687357150474625564422346765561105644    1
784164464444467676500707834543754434776884083218844016511047

In [12]:
for k in range(2, 11):
    labels = hierarchical_labels(mindist_tiebroken, k)

    print(
        k,
        labels.value_counts().values
    )

2 [671 176]
3 [613 176  58]
4 [461 176 152  58]
5 [436 176 152  58  25]
6 [436 152  90  86  58  25]
7 [436 152  90  58  46  40  25]
8 [436  90  80  72  58  46  40  25]
9 [301 135  90  80  72  58  46  40  25]
10 [301  90  88  80  72  58  47  46  40  25]


## 4. Hierarchical clustering

Fit one linkage tree from `mindist_tiebroken`, then cut it at every k in
`CANDIDATE_K_VALUES` -- cutting a fixed tree at different k is essentially
free, since scipy's `linkage()` (the expensive step) doesn't depend on k at
all. Each candidate's term assignments, cluster sizes, and dendrogram are
saved to their own `04_clustering/k_{k}/` folder, plus a side-by-side
`candidate_k_comparison.csv` for comparing structural stats at a glance.

`FINAL_K` (Step 0) selects which candidate becomes the notebook-wide primary
solution: `labels_final`, used by every downstream section (validation,
robustness, poster). Change `FINAL_K` and re-run to promote a different
candidate -- no need to touch `CANDIDATE_K_VALUES` or re-fit anything.

In [13]:
import shutil

CLUSTER_DIR = OUTPUT_DIR / "04_clustering"

if not isinstance(mindist_tiebroken, pd.DataFrame):
    raise TypeError("mindist_tiebroken must be a pandas DataFrame.")
if mindist_tiebroken.shape[0] != mindist_tiebroken.shape[1]:
    raise ValueError("mindist_tiebroken must be square.")
if not mindist_tiebroken.index.equals(mindist_tiebroken.columns):
    raise ValueError("Distance-matrix rows and columns must match in the same order.")

D_final = mindist_tiebroken.to_numpy(dtype=float)
if not np.all(np.isfinite(D_final)):
    raise ValueError("mindist_tiebroken contains NaN or infinite values.")
if not np.allclose(D_final, D_final.T, atol=1e-10):
    raise ValueError("mindist_tiebroken must be symmetric.")
if not np.allclose(np.diag(D_final), 0.0, atol=1e-10):
    raise ValueError("The diagonal of mindist_tiebroken must be zero.")

for k in CANDIDATE_K_VALUES:
    if not 2 <= k < len(mindist_tiebroken):
        raise ValueError(
            f"Every value in CANDIDATE_K_VALUES must be between 2 and "
            f"{len(mindist_tiebroken) - 1}; got k={k}."
        )

# One linkage tree for the whole notebook -- every candidate k below is just
# a different cut of this same tree, not a separate fit.
Z_SHARED = linkage(squareform(D_final, checks=False), method=LINKAGE_METHOD)


def fit_hierarchical_solution(k: int, save_dir: Path, show: bool = False) -> dict:
    """
    Cut the shared linkage tree at k, save its term assignments, cluster
    sizes, and dendrogram under `save_dir`, and return everything needed to
    interpret or compare this candidate later.
    """
    save_dir.mkdir(parents=True, exist_ok=True)

    labels = pd.Series(
        fcluster(Z_SHARED, t=k, criterion="maxclust"),
        index=mindist_tiebroken.index,
        name="cluster",
    ).astype(int)

    assignments = (
        labels.rename_axis("term").reset_index().sort_values(["cluster", "term"])
    )
    assignments.to_csv(save_dir / "cluster_assignments.csv", index=False)

    sizes = (
        labels.value_counts().sort_index()
        .rename_axis("cluster").reset_index(name="n_terms")
    )
    sizes["share_of_terms"] = sizes["n_terms"] / len(labels)
    sizes.to_csv(save_dir / "cluster_sizes.csv", index=False)

    color_threshold = Z_SHARED[-(k - 1), 2] if k > 1 else None
    fig, ax = plt.subplots(figsize=(14, 6))
    dendrogram(
        Z_SHARED, no_labels=True, color_threshold=color_threshold,
        above_threshold_color="gray", ax=ax,
    )
    ax.set_title(f"Hierarchical clustering dendrogram, k = {k}, linkage = {LINKAGE_METHOD}")
    ax.set_xlabel("Search terms")
    ax.set_ylabel("SAX MINDIST (tie-broken)")
    fig.tight_layout()
    fig.savefig(save_dir / f"dendrogram_k{k}.png", dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {"k": k, "labels": labels, "sizes": sizes}


CLUSTERING_BY_K = {
    k: fit_hierarchical_solution(k, CLUSTER_DIR / f"k_{k}", show=(k == FINAL_K))
    for k in CANDIDATE_K_VALUES
}

candidate_k_comparison = pd.DataFrame([
    {
        "k": k,
        "n_terms": len(result["labels"]),
        "min_cluster_size": int(result["sizes"]["n_terms"].min()),
        "max_cluster_size": int(result["sizes"]["n_terms"].max()),
        "median_cluster_size": float(result["sizes"]["n_terms"].median()),
        "largest_cluster_share": float(result["sizes"]["share_of_terms"].max()),
        "is_final_k": k == FINAL_K,
    }
    for k, result in CLUSTERING_BY_K.items()
])
candidate_k_comparison.to_csv(CLUSTER_DIR / "candidate_k_comparison.csv", index=False)

# Promote the configured FINAL_K to the single, notebook-wide "primary"
# solution that every downstream section (Steps 5-7) reads from here on.
labels_final = CLUSTERING_BY_K[FINAL_K]["labels"]
cluster_sizes_final = CLUSTERING_BY_K[FINAL_K]["sizes"]

# Mirror the primary solution under flat, k-independent filenames too, purely
# for discoverability -- its authoritative copy still lives in k_{FINAL_K}/.
(
    labels_final.rename_axis("term").reset_index().sort_values(["cluster", "term"])
).to_csv(CLUSTER_DIR / "final_cluster_assignments.csv", index=False)
cluster_sizes_final.to_csv(CLUSTER_DIR / "cluster_sizes_final.csv", index=False)
shutil.copy2(
    CLUSTER_DIR / f"k_{FINAL_K}" / f"dendrogram_k{FINAL_K}.png",
    CLUSTER_DIR / f"dendrogram_k{FINAL_K}.png",
)

print(f"Fit {len(CANDIDATE_K_VALUES)} candidate solutions: {CANDIDATE_K_VALUES}")
print(f"Primary (FINAL_K={FINAL_K}): {len(labels_final):,} terms across {FINAL_K} clusters.\n")
print("Candidate comparison (structural stats only -- see Step 5 for silhouette/stability):")
print(candidate_k_comparison.to_string(index=False))

Fit 5 candidate solutions: [7, 8, 9, 10, 11]
Primary (FINAL_K=10): 847 terms across 10 clusters.

Candidate comparison (structural stats only -- see Step 5 for silhouette/stability):
 k  n_terms  min_cluster_size  max_cluster_size  median_cluster_size  largest_cluster_share  is_final_k
 7      847                25               436                 58.0               0.514758       False
 8      847                25               436                 65.0               0.514758       False
 9      847                25               301                 72.0               0.355372       False
10      847                25               301                 65.0               0.355372        True
11      847                25               231                 70.0               0.272727       False


## 5. Cluster validation and interpretation

Two independent checks, both spanning multiple k:

1. **Statistical validation across the full `K_RANGE` (2-12)** -- silhouette
   and subsample stability for every k in that range, regardless of which
   ones were fit as full candidates in Step 4. Flags whether the
   stability/silhouette-recommended k agrees with `FINAL_K`, without ever
   overwriting it.
2. **Interpretive comparison across `CANDIDATE_K_VALUES`** -- term stability,
   core terms, and a cluster summary (including each cluster's MINDIST
   margin) for every candidate fit in Step 4, reusing those labels rather
   than recomputing them. `FINAL_K`'s interpretation is promoted as the
   notebook-wide primary summary that Steps 6-7 read from.

In [14]:
VALIDATION_DIR = OUTPUT_DIR / "05_validation"

def labels_from_submatrix(
    full_distance: pd.DataFrame,
    terms: list[str],
    k: int,
    method: str | None = None,
) -> pd.Series:
    method = method or LINKAGE_METHOD
    sub_distance = full_distance.loc[terms, terms]
    return hierarchical_labels(sub_distance, k=k, method=method)


def stability_for_k(
    full_distance: pd.DataFrame,
    k: int,
    n_subsamples: int,
    fraction: float,
    seed: int,
    method: str | None = None,
) -> pd.DataFrame:
    method = method or LINKAGE_METHOD
    rng = np.random.default_rng(seed)
    terms = np.asarray(full_distance.index)
    n_take = min(len(terms), max(k + 2, int(round(len(terms) * fraction))))
    rows = []

    for iteration in range(n_subsamples):
        sample_1 = rng.choice(terms, size=n_take, replace=False).tolist()
        sample_2 = rng.choice(terms, size=n_take, replace=False).tolist()
        common_terms = sorted(set(sample_1).intersection(sample_2))

        if len(common_terms) < max(3, k):
            rows.append({
                "k": k,
                "iteration": iteration,
                "n_common": len(common_terms),
                "ari": np.nan,
            })
            continue

        labels_1 = labels_from_submatrix(
            full_distance, sample_1, k, method=method
        ).loc[common_terms]
        labels_2 = labels_from_submatrix(
            full_distance, sample_2, k, method=method
        ).loc[common_terms]

        rows.append({
            "k": k,
            "iteration": iteration,
            "n_common": len(common_terms),
            "ari": adjusted_rand_score(labels_1, labels_2),
        })

    return pd.DataFrame(rows)


def validation_for_k(
    full_distance: pd.DataFrame,
    labels: pd.Series,
    k: int,
) -> dict:
    D = full_distance.to_numpy(dtype=float)
    labs = labels.loc[full_distance.index].to_numpy()
    n_observed_clusters = len(np.unique(labs))

    silhouette = (
        silhouette_score(D, labs, metric="precomputed")
        if 1 < n_observed_clusters < len(labs)
        else np.nan
    )
    sizes = labels.value_counts()

    return {
        "k": int(k),
        "n_observed_clusters": int(n_observed_clusters),
        "silhouette_mindist": float(silhouette) if pd.notna(silhouette) else np.nan,
        "min_cluster_size": int(sizes.min()),
        "max_cluster_size": int(sizes.max()),
        "median_cluster_size": float(sizes.median()),
    }


# Candidate-k validation sweep. This evaluates FINAL_K but does not replace it.
validation_rows = []
stability_tables = []

for k in K_RANGE:
    if not 2 <= k < len(mindist_tiebroken):
        continue

    labels_k = hierarchical_labels(
        mindist_tiebroken,
        k=k,
        method=LINKAGE_METHOD,
    )
    validation_rows.append(validation_for_k(mindist_tiebroken, labels_k, k))
    stability_tables.append(
        stability_for_k(
            mindist_tiebroken,
            k=k,
            n_subsamples=STABILITY_SUBSAMPLES,
            fraction=STABILITY_FRACTION,
            seed=RANDOM_STATE + k,
            method=LINKAGE_METHOD,
        )
    )

if not validation_rows:
    raise ValueError("K_RANGE contains no valid candidate values.")

validation_df = pd.DataFrame(validation_rows)
stability_raw = pd.concat(stability_tables, ignore_index=True)
stability_summary = (
    stability_raw
    .groupby("k", as_index=False)
    .agg(
        stability_mean=("ari", "mean"),
        stability_median=("ari", "median"),
        stability_p10=("ari", lambda x: x.quantile(0.10)),
        stability_p90=("ari", lambda x: x.quantile(0.90)),
        mean_common_terms=("n_common", "mean"),
        valid_stability_runs=("ari", "count"),
    )
)

cluster_selection = (
    validation_df
    .merge(stability_summary, on="k", how="left")
    .sort_values("k")
    .reset_index(drop=True)
)
cluster_selection["is_final_k"] = cluster_selection["k"].eq(FINAL_K)
cluster_selection["is_candidate_k"] = cluster_selection["k"].isin(CANDIDATE_K_VALUES)

cluster_selection.to_csv(
    VALIDATION_DIR / "cluster_selection_by_k.csv",
    index=False,
)
stability_raw.to_csv(
    VALIDATION_DIR / "subsample_stability_raw.csv",
    index=False,
)

print("Candidate cluster validation:")
print(
    cluster_selection
    .sort_values(
        ["stability_median", "silhouette_mindist"],
        ascending=[False, False],
        na_position="last",
    )
    .to_string(index=False)
)


def select_recommended_k(
    selection_df: pd.DataFrame,
    min_cluster_size: int,
) -> int:
    eligible = selection_df[
        selection_df["min_cluster_size"] >= min_cluster_size
    ]
    if eligible.empty:
        eligible = selection_df
    best = eligible.sort_values(
        ["stability_median", "silhouette_mindist"],
        ascending=[False, False],
        na_position="last",
    ).iloc[0]
    return int(best["k"])


RECOMMENDED_K = select_recommended_k(
    cluster_selection,
    MIN_ACCEPTABLE_CLUSTER_SIZE,
)

k_selection_audit = pd.DataFrame({
    "metric": [
        "configured_final_k",
        "recommended_k",
        "min_acceptable_cluster_size",
        "configured_k_matches_recommendation",
    ],
    "value": [
        FINAL_K,
        RECOMMENDED_K,
        MIN_ACCEPTABLE_CLUSTER_SIZE,
        FINAL_K == RECOMMENDED_K,
    ],
})

k_selection_audit.to_csv(
    VALIDATION_DIR / "k_selection_audit.csv",
    index=False,
)

print(
    f"\nConfigured FINAL_K={FINAL_K}; validation recommends "
    f"k={RECOMMENDED_K}. The configured baseline is not changed."
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(
    cluster_selection["k"],
    cluster_selection["stability_median"],
    marker="o",
    label="Median subsample stability",
)
ax.plot(
    cluster_selection["k"],
    cluster_selection["silhouette_mindist"],
    marker="s",
    label="Silhouette score",
)
ax.axvline(
    FINAL_K,
    linestyle="--",
    linewidth=1.5,
    label=f"Final k = {FINAL_K}",
)
ax.set_title("Cluster validation across candidate values of k")
ax.set_xlabel("Number of clusters, k")
ax.set_ylabel("Validation score")
ax.set_xticks(cluster_selection["k"])
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(
    VALIDATION_DIR / "cluster_selection_scores.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


def term_stability_from_distance(
    full_distance: pd.DataFrame,
    final_labels: pd.Series,
) -> pd.DataFrame:
    rows = []
    aligned_labels = final_labels.loc[full_distance.index]

    for term in aligned_labels.index:
        cluster_id = aligned_labels.loc[term]
        same_terms = aligned_labels.index[
            (aligned_labels == cluster_id) & (aligned_labels.index != term)
        ]
        other_terms = aligned_labels.index[aligned_labels != cluster_id]

        intra_dist = (
            full_distance.loc[term, same_terms].mean()
            if len(same_terms) > 0
            else np.nan
        )
        between_dist = (
            full_distance.loc[term, other_terms].mean()
            if len(other_terms) > 0
            else np.nan
        )
        margin = (
            between_dist - intra_dist
            if pd.notna(intra_dist) and pd.notna(between_dist)
            else np.nan
        )

        rows.append({
            "term": term,
            "cluster": int(cluster_id),
            "mean_intra_mindist": intra_dist,
            "mean_between_mindist": between_dist,
            "mindist_margin": margin,
        })

    term_df = pd.DataFrame(rows)
    term_df["term_stability"] = (
        term_df
        .groupby("cluster")["mindist_margin"]
        .rank(pct=True, method="average")
    )
    return term_df.sort_values(
        ["cluster", "term_stability", "mindist_margin"],
        ascending=[True, False, False],
    ).reset_index(drop=True)


def select_core_terms(
    term_stability: pd.DataFrame,
    top_n: int = TOP_N_CORE_TERMS,
) -> pd.DataFrame:
    selected = []

    for cluster_id, group in term_stability.groupby("cluster"):
        group = group.sort_values(
            ["term_stability", "mindist_margin"],
            ascending=[False, False],
        ).copy()
        cutoff = group["term_stability"].quantile(CORE_STABILITY_QUANTILE)
        core = group[group["term_stability"] >= cutoff].copy()

        min_keep = min(MIN_CORE_TERMS, len(group))
        if len(core) < min_keep:
            core = group.head(min_keep).copy()
        if top_n is not None and top_n > 0:
            core = core.head(max(top_n, min_keep)).copy()

        core["core_rank"] = np.arange(1, len(core) + 1)
        selected.append(core)

    return pd.concat(selected, ignore_index=True)


def weighted_average(panel: pd.DataFrame, weights: pd.Series) -> pd.Series:
    columns = [column for column in weights.index if column in panel.columns]
    if not columns:
        return pd.Series(np.nan, index=panel.index, dtype=float)

    aligned_weights = weights.loc[columns].astype(float).fillna(0.0)
    if aligned_weights.sum() <= 0:
        aligned_weights = pd.Series(1.0 / len(columns), index=columns)
    else:
        aligned_weights = aligned_weights / aligned_weights.sum()

    return panel[columns].mul(aligned_weights, axis=1).sum(axis=1)


def interpret_solution(
    k: int,
    labels: pd.Series,
    save_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build term-level stability, core terms, and a cluster summary for one
    already-fitted candidate k (from Step 4's CLUSTERING_BY_K), without
    recomputing its cluster assignments.
    """
    save_dir.mkdir(parents=True, exist_ok=True)

    term_stability_k = term_stability_from_distance(mindist, labels)
    core_terms_k = select_core_terms(term_stability_k, top_n=TOP_N_CORE_TERMS)

    term_stability_k.to_csv(save_dir / "term_stability.csv", index=False)
    core_terms_k.to_csv(save_dir / "core_terms.csv", index=False)

    summary_k = (
        term_stability_k
        .groupby("cluster", as_index=False)
        .agg(
            n_terms=("term", "size"),
            median_term_stability=("term_stability", "median"),
            mean_intra_mindist=("mean_intra_mindist", "mean"),
            mean_between_mindist=("mean_between_mindist", "mean"),
            mean_mindist_margin=("mindist_margin", "mean"),
        )
    )
    top_terms_k = (
        core_terms_k
        .sort_values(["cluster", "core_rank"])
        .groupby("cluster")["term"]
        .apply(lambda terms: ", ".join(terms.head(TOP_N_CORE_TERMS)))
    )
    summary_k["top_10_core_terms"] = summary_k["cluster"].map(top_terms_k)

    expected_sizes = labels.value_counts().sort_index()
    summary_k["expected_n_terms"] = summary_k["cluster"].map(expected_sizes)
    summary_k["size_matches_labels"] = (
        summary_k["n_terms"] == summary_k["expected_n_terms"]
    )
    if not summary_k["size_matches_labels"].all():
        raise RuntimeError(f"Cluster-summary sizes do not match labels for k={k}.")

    summary_k.to_csv(save_dir / "cluster_summary.csv", index=False)
    return term_stability_k, core_terms_k, summary_k


# Interpret every candidate fit in Step 4 -- reuses CLUSTERING_BY_K, so no
# cluster assignment is recomputed here, only its downstream interpretation.
INTERPRETATION_BY_K = {
    k: interpret_solution(
        k, CLUSTERING_BY_K[k]["labels"], VALIDATION_DIR / f"k_{k}"
    )
    for k in CANDIDATE_K_VALUES
}

candidate_k_interpretation = pd.DataFrame([
    {
        "k": k,
        "n_clusters": len(summary_k),
        "weakest_cluster_margin": float(summary_k["mean_mindist_margin"].min()),
        "strongest_cluster_margin": float(summary_k["mean_mindist_margin"].max()),
        "largest_cluster_n_terms": int(summary_k["n_terms"].max()),
        "is_final_k": k == FINAL_K,
    }
    for k, (_, _, summary_k) in INTERPRETATION_BY_K.items()
])
candidate_k_interpretation.to_csv(
    VALIDATION_DIR / "candidate_k_interpretation_summary.csv", index=False
)

print("Interpretive comparison across candidate k values:")
print(
    "(weakest_cluster_margin close to 0 or negative flags a cluster that is "
    "not well separated -- see cluster_summary.csv in that k's folder)\n"
)
print(candidate_k_interpretation.to_string(index=False))

# Promote FINAL_K's interpretation as the notebook-wide primary summary, and
# mirror it under flat, k-independent filenames for discoverability -- its
# authoritative copy still lives in k_{FINAL_K}/.
term_stability, core_terms, cluster_summary = INTERPRETATION_BY_K[FINAL_K]
term_stability.to_csv(VALIDATION_DIR / "term_stability.csv", index=False)
core_terms.to_csv(VALIDATION_DIR / "core_terms.csv", index=False)
cluster_summary.to_csv(VALIDATION_DIR / "cluster_summary.csv", index=False)

print(f"\nPrimary cluster interpretation (FINAL_K={FINAL_K}):")
print(cluster_summary.to_string(index=False))

Candidate cluster validation:
 k  n_observed_clusters  silhouette_mindist  min_cluster_size  max_cluster_size  median_cluster_size  stability_mean  stability_median  stability_p10  stability_p90  mean_common_terms  valid_stability_runs  is_final_k  is_candidate_k
 3                    3            0.060385                58               613                176.0        0.382001          0.387509       0.152782       0.614605           542.3375                    80       False           False
 8                    8            0.035004                25               436                 65.0        0.395318          0.379658       0.294352       0.503064           543.6125                    80       False            True
 9                    9            0.042690                25               301                 72.0        0.360076          0.357238       0.294452       0.434958           544.0375                    80       False            True
12                   12           

## 6. Robustness and sensitivity analysis

This section holds the Step 4 baseline partition fixed and tests whether the conclusions depend on modeling choices. It includes: PAA distribution diagnostics, MINDIST tightness, linkage-method comparison, PAA reconstruction ablation, downstream `(w, a)` quality, clipping robustness, and consolidated one-factor-at-a-time sensitivity analysis.

**Crucial outputs:** the no-change OFAAT control, ARI versus the fixed baseline, surviving-term counts, linkage stability, the MINDIST lower-bound check, and clustering stability across clipping and representation choices.

In [15]:
ROBUSTNESS_DIR = OUTPUT_DIR / "06_robustness"

# The Gaussian-vs-empirical breakpoint fit diagnostic (excess kurtosis, KS
# test, QQ-plot) already ran once in Section 2 and was saved to
# paa_gaussian_fit_diagnostic.{csv,png} in this same directory -- see that
# cell for the result; it is not recomputed here.

In [16]:
def mindist_tightness_check(
    panel: pd.DataFrame,
    segment_grid: list[int],
    alphabet_grid: list[int],
    sample_size: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    cols = panel.columns.to_list()
    sample_cols = rng.choice(cols, size=min(sample_size, len(cols)), replace=False)
    sub = panel[sample_cols]

    X = panel_to_tslearn_array(sub)
    n_timestamps = sub.shape[0]
    flat = X[:, :, 0]
    euclid = squareform(pdist(flat, metric="euclidean"))
    mask = ~np.eye(len(sample_cols), dtype=bool)

    rows = []
    for w in segment_grid:
        paa_w = PiecewiseAggregateApproximation(n_segments=w).fit_transform(X)[:, :, 0]
        for a in alphabet_grid:
            bp = (
                empirical_breakpoints(paa_w.ravel(), a)
                if BREAKPOINT_MODE == "empirical"
                else sax_breakpoints(a)
            )
            codes = discretize_paa(paa_w, bp)
            sd = sax_symbol_distance_table(a, bp)
            scale = np.sqrt(n_timestamps / w)
            per_seg = sd[codes[:, None, :], codes[None, :, :]]
            md = scale * np.sqrt((per_seg ** 2).sum(axis=2))

            ratio = md[mask] / np.clip(euclid[mask], 1e-9, None)
            rows.append({
                "n_segments": w,
                "alphabet_size": a,
                "mean_tightness_ratio": float(np.mean(ratio)),
                "median_tightness_ratio": float(np.median(ratio)),
                "share_mindist_exceeds_euclid": float(np.mean(md[mask] > euclid[mask] + 1e-6)),
            })
    return pd.DataFrame(rows)


tightness_df = mindist_tightness_check(
    panel_sax, SEGMENT_GRID, ALPHABET_GRID, ABLATION_SAMPLE_SIZE, RANDOM_STATE
)
tightness_df.to_csv(ROBUSTNESS_DIR / "mindist_tightness_by_w_a.csv", index=False)

chosen_row = tightness_df[
    (tightness_df["n_segments"] == N_SEGMENTS) & (tightness_df["alphabet_size"] == ALPHABET_SIZE)
]
print("(w, a) tightness ablation (higher mean_tightness_ratio = less information lost):")
print(tightness_df.sort_values("mean_tightness_ratio", ascending=False).to_string(index=False))
print("\nCurrently configured (N_SEGMENTS, ALPHABET_SIZE) row:")
print(chosen_row.to_string(index=False))

(w, a) tightness ablation (higher mean_tightness_ratio = less information lost):
 n_segments  alphabet_size  mean_tightness_ratio  median_tightness_ratio  share_mindist_exceeds_euclid
        128             11              0.388816                0.391334                           0.0
         96             11              0.360789                0.361860                           0.0
        128              9              0.348101                0.350331                           0.0
         96              9              0.322465                0.323694                           0.0
         64             11              0.303398                0.303838                           0.0
        128              7              0.292205                0.293450                           0.0
         64              9              0.273833                0.274399                           0.0
         96              7              0.273356                0.273721                       

In [17]:
_k_for_linkage_check = FINAL_K

linkage_rows = []
for method in LINKAGE_CANDIDATES:
    labels_m = hierarchical_labels(mindist_tiebroken, _k_for_linkage_check, method=method)
    val = validation_for_k(mindist_tiebroken, labels_m, _k_for_linkage_check)
    stab = stability_for_k(
        mindist_tiebroken, _k_for_linkage_check, STABILITY_SUBSAMPLES, STABILITY_FRACTION, RANDOM_STATE, method=method
    )
    linkage_rows.append({
        "linkage_method": method,
        "k_used": _k_for_linkage_check,
        "silhouette_mindist": val["silhouette_mindist"],
        "min_cluster_size": val["min_cluster_size"],
        "max_cluster_size": val["max_cluster_size"],
        "stability_mean": stab["ari"].mean(),
        "stability_median": stab["ari"].median(),
    })

linkage_comparison = pd.DataFrame(linkage_rows)
linkage_comparison.to_csv(ROBUSTNESS_DIR / "linkage_method_comparison.csv", index=False)
print(f"Linkage method comparison at k={_k_for_linkage_check} (currently configured LINKAGE_METHOD = '{LINKAGE_METHOD}'):")
print(linkage_comparison.sort_values("stability_median", ascending=False).to_string(index=False))

# The stability/silhouette-recommended k (RECOMMENDED_K) and its comparison
# against FINAL_K were already computed and logged in Section 5
# (k_selection_audit.csv); not recomputed here.
print(
    f"\nValidation-recommended k (from Section 5): {RECOMMENDED_K}; "
    f"configured baseline FINAL_K: {FINAL_K}. Step 6 does not alter the baseline partition."
)

Linkage method comparison at k=10 (currently configured LINKAGE_METHOD = 'ward'):
linkage_method  k_used  silhouette_mindist  min_cluster_size  max_cluster_size  stability_mean  stability_median
        single      10            0.050663                 1               835        0.809724          0.819579
       average      10            0.059943                 1               828        0.654552          0.659066
          ward      10            0.040294                25               301        0.367501          0.361258
      complete      10            0.028313                 8               284        0.211981          0.215635

Validation-recommended k (from Section 5): 3; configured baseline FINAL_K: 10. Step 6 does not alter the baseline partition.


### Robustness test on (w, a) and clip-threshold

The tightness ablation above always favors larger `(w, a)` -- it can show
`(96, 9)` is *tighter than* smaller grid points, but not that it's *sufficient*,
since tightness improves monotonically with more segments/symbols right up to
the trivial `w = n` limit. The three checks below answer the sufficiency
question directly instead:

1. **PAA reconstruction elbow** -- where does adding more segments stop
   meaningfully reducing reconstruction error, independent of clustering?
2. **Downstream quality by (w, a)** -- does the actual clustering result
   (stability, silhouette) change if `N_SEGMENTS` / `ALPHABET_SIZE` are
   increased beyond the configured values?
3. **Clip-threshold sweep** -- does the `±5` clipping bound sit on a
   plateau where both the clipped share and the resulting clustering have
   stopped changing, or is it an arbitrary cutoff?

None of these prove `(96, 9, ±5)` is uniquely optimal -- only that it is a
defensible point on a diminishing-returns curve, which is what gets reported.

In [18]:
# -----------------------------------------------------------------------------
# 1. PAA reconstruction-error elbow vs. n_segments
# -----------------------------------------------------------------------------
def paa_reconstruction_error(X: np.ndarray, w: int) -> float:
    """Mean squared error between each series and its own w-segment PAA reconstruction."""
    n_terms, n_timestamps, _ = X.shape
    paa_coefs = PiecewiseAggregateApproximation(n_segments=w).fit_transform(X)[:, :, 0]
    seg_len = n_timestamps // w
    reconstructed = np.repeat(paa_coefs, seg_len, axis=1)
    if reconstructed.shape[1] < n_timestamps:
        pad = n_timestamps - reconstructed.shape[1]
        reconstructed = np.concatenate([reconstructed, reconstructed[:, -1:].repeat(pad, axis=1)], axis=1)
    return float(np.mean((X[:, :, 0] - reconstructed[:, :n_timestamps]) ** 2))


elbow_grid = sorted(set(SEGMENT_GRID + [panel_norm.shape[0] // r for r in (2, 4, 8, 16, 32)]))
elbow_grid = [w for w in elbow_grid if 4 <= w <= panel_norm.shape[0]]

recon_errors = pd.DataFrame({
    "n_segments": elbow_grid,
    "reconstruction_mse": [paa_reconstruction_error(X_ts, w) for w in elbow_grid],
})
recon_errors["compression_ratio"] = panel_norm.shape[0] / recon_errors["n_segments"]
# marginal gain: how much MSE drops per additional segment, normalized -- the
# elbow is where this flattens out rather than where MSE is simply lowest.
recon_errors["marginal_gain_per_segment"] = (
    -recon_errors["reconstruction_mse"].diff() / recon_errors["n_segments"].diff()
)
recon_errors.to_csv(ROBUSTNESS_DIR / "paa_reconstruction_elbow.csv", index=False)
print("PAA reconstruction error and marginal gain per added segment:")
print(recon_errors.to_string(index=False))

PAA reconstruction error and marginal gain per added segment:
 n_segments  reconstruction_mse  compression_ratio  marginal_gain_per_segment
         48            2.666653          33.583333                        NaN
         50            2.628020          32.240000                   0.019317
         64            2.441669          25.187500                   0.013311
         96            2.196801          16.791667                   0.007652
        100            2.055099          16.120000                   0.035426
        128            1.997574          12.593750                   0.002054
        201            1.507233           8.019900                   0.006717
        403            0.867792           4.000000                   0.003166
        806            0.306019           2.000000                   0.001394


In [19]:
# -----------------------------------------------------------------------------
# 2. Downstream clustering quality vs. (w, a)
# Tightness always improves with more segments/symbols; this checks whether
# the actual clustering ANSWER (stability, silhouette) improves too, at the
# fixed baseline k from Step 4.
# -----------------------------------------------------------------------------
def downstream_quality_for_wa(
    panel: pd.DataFrame,
    w: int,
    a: int,
    k: int,
    n_subsamples: int = 20,
) -> dict:
    X = panel_to_tslearn_array(panel)
    paa_w = PiecewiseAggregateApproximation(n_segments=w).fit_transform(X)[:, :, 0]
    bp = empirical_breakpoints(paa_w.ravel(), a) if BREAKPOINT_MODE == "empirical" else sax_breakpoints(a)
    codes = discretize_paa(paa_w, bp)
    sd = sax_symbol_distance_table(a, bp)
    scale = np.sqrt(panel.shape[0] / w)
    per_seg = sd[codes[:, None, :], codes[None, :, :]]
    D = scale * np.sqrt((per_seg ** 2).sum(axis=2))
    D = pd.DataFrame(D, index=panel.columns, columns=panel.columns)

    # Match the baseline procedure: resolve SAX quantization ties with the
    # full-resolution Euclidean distance before clustering.
    euclid = true_euclidean_distance_matrix(panel)
    D_tiebroken = break_mindist_ties(D, euclid, eps=TIE_EPS)

    labels_k = hierarchical_labels(D_tiebroken, k, method=LINKAGE_METHOD)
    val = validation_for_k(D_tiebroken, labels_k, k)
    stab = stability_for_k(
        D_tiebroken, k, n_subsamples, STABILITY_FRACTION, RANDOM_STATE,
        method=LINKAGE_METHOD,
    )
    return {
        "n_segments": w, "alphabet_size": a,
        "silhouette_mindist": val["silhouette_mindist"],
        "min_cluster_size": val["min_cluster_size"],
        "stability_median": stab["ari"].median(),
    }


downstream_rows = [
    downstream_quality_for_wa(panel_sax, w, a, k=FINAL_K)
    for w in SEGMENT_GRID for a in ALPHABET_GRID
]
downstream_df = pd.DataFrame(downstream_rows)
downstream_df.to_csv(ROBUSTNESS_DIR / "downstream_quality_by_w_a.csv", index=False)
print(f"Downstream clustering quality by (w, a) at k={FINAL_K}, linkage={LINKAGE_METHOD}:")
print(downstream_df.sort_values("stability_median", ascending=False).to_string(index=False))

Downstream clustering quality by (w, a) at k=10, linkage=ward:
 n_segments  alphabet_size  silhouette_mindist  min_cluster_size  stability_median
         48             11            0.070929                32          0.388116
         96              9            0.040294                25          0.384657
         48              7            0.049061                27          0.378434
         48              9            0.073054                26          0.376962
        128             11            0.028486                24          0.374378
        128              9            0.012040                27          0.371060
        128              7            0.013995                25          0.365846
         64             11            0.032466                26          0.362099
         64              5            0.058882                37          0.361856
         96              7            0.038668                31          0.361253
         48             

In [20]:
# -----------------------------------------------------------------------------
# 3. Clip-threshold robustness sweep
#
# Compare every clipping solution with an explicitly computed unclipped
# reference. Each solution also uses the same Euclidean tie-breaking procedure
# as the Step 4 baseline.
# -----------------------------------------------------------------------------
CLIP_GRID = [2, 3, 4, 5, 6, 8, np.inf]
raw_vals = panel_norm.to_numpy(dtype=float)

labels_by_clip = {}
share_clipped_by_clip = {}

for clip_threshold in CLIP_GRID:
    clipped = (
        panel_norm.clip(-clip_threshold, clip_threshold)
        if np.isfinite(clip_threshold)
        else panel_norm.copy()
    )

    share_clipped_by_clip[clip_threshold] = (
        float(np.mean(np.abs(raw_vals) > clip_threshold))
        if np.isfinite(clip_threshold)
        else 0.0
    )

    X_c = panel_to_tslearn_array(clipped)
    paa_c = PiecewiseAggregateApproximation(
        n_segments=N_SEGMENTS
    ).fit_transform(X_c)[:, :, 0]

    bp_c = (
        empirical_breakpoints(paa_c.ravel(), ALPHABET_SIZE)
        if BREAKPOINT_MODE == "empirical"
        else sax_breakpoints(ALPHABET_SIZE)
    )
    codes_c = discretize_paa(paa_c, bp_c)
    sd_c = sax_symbol_distance_table(ALPHABET_SIZE, bp_c)
    scale_c = np.sqrt(clipped.shape[0] / N_SEGMENTS)
    per_seg_c = sd_c[codes_c[:, None, :], codes_c[None, :, :]]

    D_c = pd.DataFrame(
        scale_c * np.sqrt((per_seg_c ** 2).sum(axis=2)),
        index=clipped.columns,
        columns=clipped.columns,
    )
    euclid_c = true_euclidean_distance_matrix(clipped)
    D_c_tiebroken = break_mindist_ties(D_c, euclid_c, eps=TIE_EPS)

    labels_by_clip[clip_threshold] = hierarchical_labels(
        D_c_tiebroken,
        FINAL_K,
        method=LINKAGE_METHOD,
    )

unclipped_labels = labels_by_clip[np.inf]
clip_rows = []

for clip_threshold in CLIP_GRID:
    labels_c = labels_by_clip[clip_threshold]
    common_terms = unclipped_labels.index.intersection(labels_c.index)
    ari_vs_unclipped = adjusted_rand_score(
        unclipped_labels.loc[common_terms],
        labels_c.loc[common_terms],
    )

    clip_rows.append({
        "clip_threshold": (
            clip_threshold if np.isfinite(clip_threshold) else "none"
        ),
        "share_clipped": share_clipped_by_clip[clip_threshold],
        "ari_vs_unclipped": float(ari_vs_unclipped),
        "n_common_terms": int(len(common_terms)),
    })

clip_df = pd.DataFrame(clip_rows)
clip_df.to_csv(
    ROBUSTNESS_DIR / "clip_threshold_sweep.csv",
    index=False,
)
print(f"Clip-threshold sweep at FINAL_K={FINAL_K}, linkage={LINKAGE_METHOD}:")
print(clip_df.to_string(index=False))


Clip-threshold sweep at FINAL_K=10, linkage=ward:
clip_threshold  share_clipped  ari_vs_unclipped  n_common_terms
             2       0.186075          0.260999             847
             3       0.124327          0.257544             847
             4       0.092994          0.304608             847
             5       0.075727          0.278795             847
             6       0.063929          0.323991             847
             8       0.049133          0.282659             847
          none       0.000000          1.000000             847


### Consolidated sensitivity analysis (One-Factor-At-A-Time)

**Why this check exists.** Hierarchical clustering has no ground-truth
partition to validate against, so the only defensible evidence for "these
clusters are real" is that they survive small, individually reasonable
perturbations to hyperparameters that were otherwise chosen by convention
(a denoise window, a filtering threshold, a date boundary) rather than
derived from theory. Absent this check, a reviewer is entitled to ask
whether the reported clusters are an artifact of one specific configuration
-- this section answers that question directly, one parameter at a time,
and reports the answer as a single Adjusted Rand Index (ARI) per parameter
against the baseline partition.

**Design: One-Factor-At-A-Time (OFAAT), not a full factorial or global
sensitivity design.** Each parameter is perturbed independently, holding
every other parameter at its baseline value, following the standard
elementary-effects screening approach described in Saltelli et al.
(*Global Sensitivity Analysis: The Primer*, Wiley, 2008). OFAAT is
computationally cheap and directly interpretable ("does *this* choice
matter?"), but it cannot detect interaction effects between parameters
(e.g., a joint change in `N_SEGMENTS` and `ALPHABET_SIZE` that individually
look fine but compound). A full Sobol-index or Morris-method design would
capture interactions but requires an order of magnitude more pipeline
reruns than is practical here, since several perturbations require
reloading and reprocessing ~850 raw time series from disk. OFAAT is the
right tool for "is any single hardcoded choice load-bearing," which is the
question a methods reviewer is most likely to raise.

**Why Adjusted Rand Index, not raw agreement or the plain Rand Index.**
The Rand Index (Rand, 1971) counts the share of point-pairs the two
partitions agree on, but its expected value under random labeling grows
with the number of clusters and is not zero -- so two independent random
partitions of ~850 terms into ~10 clusters can score deceptively high on
raw agreement alone. Hubert & Arabie's Adjusted Rand Index
(*Comparing Partitions*, Journal of Classification, 1985) corrects for this
by subtracting the expected value under the generalized hypergeometric null
model and rescaling so that 0 indicates chance-level agreement and 1
indicates identical partitions. This is why ARI, not accuracy or raw pair
agreement, is the standard metric for exactly this kind of clustering
comparison task.

**Row-Discrepancy Trap: intersection alignment.** Perturbing a filtering
threshold (`MAX_ZERO_SHARE`, `MAX_MISSING_SHARE`) or the analysis window
(`START_DATE`, `END_DATE`) changes *which* terms survive Section 1's
filtering funnel, so the perturbed run's label vector can have a different
length and a different index than the baseline. `adjusted_rand_score`
requires two positionally-aligned, equal-length arrays -- it does not match
by label -- so passing misaligned series either raises or silently compares
the wrong pairs of terms. The comparison here is therefore restricted to the
**intersection** of the baseline and perturbed term sets, both explicitly
re-indexed to an identical sorted order before scoring. This is the standard
approach for comparing partitions defined over non-identical object sets
(see Vinh, Epps & Bailey, *Information Theoretic Measures for Clusterings
Comparison*, JMLR, 2010, for the general treatment of clustering-comparison
measures under such conditions). Because a high ARI computed over a small
intersection is much weaker evidence than the same ARI over a nearly
complete one, **`Surviving N` is reported alongside every ARI value** and
should be read jointly with it, not the ARI alone.

**Fixed-$K$ constraint.** Every perturbed run is sliced at exactly
`BASELINE_K` clusters (the $K$ that produced the reported baseline
partition), regardless of what a fresh stability sweep on the perturbed
distance matrix might independently prefer. This deliberately isolates one
question -- *"given the same number of groups, do the same terms end up
together?"* -- from a different question -- *"would a different number of
groups be preferred under this perturbation?"* -- which the existing
`cluster_selection_by_k.csv` / `k_selection_log.csv` already address
separately in Section 4 and above. Conflating the two would make it
impossible to tell whether a low ARI reflects genuine membership
instability or merely a different (but possibly equally valid) preferred
tree cut.

**Why Ward linkage, restated with the evidence now on hand.** Three
independent lines of justification converge on Ward's method as the
right default for this pipeline, not just the axiomatic one raised earlier
in this conversation:

1. *Axiomatic:* Kleinberg's impossibility theorem (*An Impossibility Theorem
   for Clustering*, NeurIPS, 2002) shows no partition-valued clustering
   function can jointly satisfy scale invariance, richness, and consistency;
   Zadeh & Ben-David (*A Uniqueness Theorem for Clustering*, UAI, 2009)
   resolve this at the dendrogram level and show single linkage is the
   unique method satisfying the corresponding tree-valued axioms. This
   motivates single linkage as a theoretically privileged *default*, not a
   guarantee for any specific dataset.
2. *Empirical, on this exact MINDIST matrix:* `linkage_method_comparison.csv`
   (computed earlier in this section) shows single and average linkage
   collapse into one dominant cluster plus a long tail of singletons on this
   panel -- the classic chaining failure mode -- while Ward and complete
   linkage recover non-trivial, comparably-sized partitions. Between those
   two, Ward scored consistently higher on both subsample stability and
   silhouette across every $k$ tested. The theoretical default was
   empirically falsified for this dataset and replaced with the
   better-performing alternative, which is the correct response to a
   theory-practice conflict.
3. *Structural fit:* Ward's method (Ward, *Hierarchical Grouping to
   Optimize an Objective Function*, JASA, 1963; see Murtagh & Legendre,
   *Ward's Hierarchical Agglomerative Clustering Method: Which Algorithms
   Implement Ward's Criterion?*, Journal of Classification, 2014, for the
   modern Lance-Williams formulation) merges the pair of clusters that
   minimizes the resulting increase in within-cluster sum of squared
   distances. MINDIST is itself defined as a scaled root-sum-of-squares
   over symbol-distance terms, so a sum-of-squares-minimizing merge
   criterion is a natural match to the geometry of the distance being
   clustered -- unlike, for instance, a criterion built around distance
   maxima (complete linkage) or a non-Euclidean-flavored distance where
   Ward's Lance-Williams update is less well motivated. This is also
   consistent with the broader empirical finding, across many simulated and
   real datasets, that Ward's method tends to recover true cluster
   structure more reliably than single, average, or centroid linkage
   (Milligan & Cooper, *An Examination of Procedures for Determining the
   Number of Clusters in a Data Set*, Psychometrika, 1985).

**How to read the table.** An ARI at or above roughly 0.80 is conventionally
treated as strong partition agreement in the clustering-comparison
literature; values well below that, particularly alongside a `Surviving N`
close to the full baseline count (so the drop isn't just an artifact of
term attrition), indicate the parameter is load-bearing and its choice
should be justified explicitly in the writeup rather than left as an
unexamined default. As with any OFAAT screen, a table that is uniformly
high does not *prove* global robustness (interaction effects are untested
by construction) -- it demonstrates the absence of first-order fragility,
which is the claim actually being made here.

In [21]:
# -----------------------------------------------------------------------------
# Consolidated OFAAT sensitivity analysis (Refactored)
# Inputs already defined: DATA_DIR, panel_raw, panel_norm, labels_final,
# FINAL_K, build_panel, preprocess_panel, panel_to_tslearn_array,
# sax_breakpoints, empirical_breakpoints, discretize_paa,
# sax_symbol_distance_table, sax_mindist_matrix, true_euclidean_distance_matrix,
# break_mindist_ties, hierarchical_labels (Steps 1-5)
# -----------------------------------------------------------------------------
import copy
from contextlib import contextmanager
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import numpy as np

# --- Baseline snapshot: partition and K being perturbed against ------------
BASELINE_LABELS = labels_final.copy()
BASELINE_K = FINAL_K

# --- Perturbation spec: one reasonable alternative per parameter -----------
PERTURBATIONS = {
    "Preprocessing": {
        "DENOISE_WINDOW": [7, 15, 21],
        "DENOISE_POLYORDER": [2, 3],
        "DETREND_WINDOW": [31, 61, 91],
    },

    "Filtering": {
        "MAX_ZERO_SHARE": [0.30, 0.45, 0.60],
        "MAX_MISSING_SHARE": [0.02, 0.05, 0.10],
        "MAX_INTERPOLATE_GAP": [7, 14, 21],
    },

    "Representation": {
        "BREAKPOINT_MODE": ["empirical", "gaussian"],
        "LINKAGE_METHOD": ["average", "complete", "ward"],
        "TIE_EPS": [1e-6, 1e-5, 1e-4],
    },
}

STAGE_RERUN_TIER = {
    "Preprocessing": "preprocessing",
    "Filtering": "filtering",
    "Representation": "representation"
}

BASELINE_CONFIG = {
    param: globals()[param] for params in PERTURBATIONS.values() for param in params
}

@contextmanager
def isolated_global_override(param: str, value):
    """
    Temporarily override one notebook-level configuration parameter.

    The original value is restored even if the perturbed pipeline run fails.
    """
    if param not in globals():
        raise KeyError(
            f"Cannot override {param!r}: the parameter is not defined globally."
        )

    original_value = copy.deepcopy(globals()[param])

    try:
        globals()[param] = value
        yield
    finally:
        globals()[param] = original_value


def run_pipeline_proxy(rerun_from: str) -> pd.Series:
    """
    Run a deterministic proxy of the clustering pipeline under the currently
    active global configuration.

    Parameters
    ----------
    rerun_from:
        Earliest stage that must be recomputed.

        "filtering"
            Rebuild the panel from disk because filtering thresholds are
            applied inside build_panel(), then preprocess, encode, and cluster.

        "preprocessing"
            Reuse the baseline filtered raw panel, then rerun preprocessing,
            encoding, and clustering.

        "representation"
            Reuse the finalized baseline normalized panel, then rerun SAX,
            distance construction, and clustering.

    Returns
    -------
    pd.Series
        Cluster labels indexed by the terms surviving the perturbed run.
    """
    valid_tiers = {
        "filtering",
        "preprocessing",
        "representation",
    }

    if rerun_from not in valid_tiers:
        raise ValueError(
            f"Unsupported rerun tier {rerun_from!r}. "
            f"Expected one of {sorted(valid_tiers)}."
        )

    # Read active configuration values at execution time so temporary global
    # overrides are reflected in each OFAAT run.
    current_n_segments = int(N_SEGMENTS)
    current_alphabet_size = int(ALPHABET_SIZE)
    current_breakpoint_mode = str(BREAKPOINT_MODE)
    current_tie_eps = float(TIE_EPS)
    current_linkage_method = str(LINKAGE_METHOD)

    if current_n_segments < 1:
        raise ValueError("N_SEGMENTS must be at least 1.")

    if current_alphabet_size < 2:
        raise ValueError("ALPHABET_SIZE must be at least 2.")

    if current_breakpoint_mode not in {"empirical", "gaussian"}:
        raise ValueError(
            "BREAKPOINT_MODE must be either 'empirical' or 'gaussian'."
        )

    # -------------------------------------------------------------------------
    # Recompute the required upstream stages
    # -------------------------------------------------------------------------

    if rerun_from == "filtering":
        # build_panel() dynamically reads MAX_ZERO_SHARE,
        # MAX_MISSING_SHARE, and MAX_INTERPOLATE_GAP.
        panel_raw_p, _ = build_panel(DATA_DIR)
        panel_norm_p, _ = preprocess_panel(panel_raw_p)

    elif rerun_from == "preprocessing":
        # Reuse the baseline panel after filtering, but rerun interpolation,
        # denoising, detrending, and robust normalization.
        panel_norm_p, _ = preprocess_panel(panel_raw.copy())

    else:
        # Representation-only changes reuse the fixed baseline normalized panel.
        panel_norm_p = panel_norm.copy()

    if not isinstance(panel_norm_p, pd.DataFrame):
        raise TypeError(
            "The proxy pipeline must produce a pandas DataFrame."
        )

    if panel_norm_p.empty:
        raise ValueError(
            f"The perturbed pipeline produced an empty panel at "
            f"rerun tier {rerun_from!r}."
        )

    if panel_norm_p.shape[1] < BASELINE_K:
        raise ValueError(
            f"Only {panel_norm_p.shape[1]} terms survived, fewer than "
            f"BASELINE_K={BASELINE_K}."
        )

    if current_n_segments > panel_norm_p.shape[0]:
        raise ValueError(
            f"N_SEGMENTS={current_n_segments} exceeds the number of "
            f"timestamps ({panel_norm_p.shape[0]})."
        )

    # -------------------------------------------------------------------------
    # SAX representation
    # -------------------------------------------------------------------------

    panel_sax_p = panel_norm_p.clip(-5, 5)
    X_ts_p = panel_to_tslearn_array(panel_sax_p)

    paa_p = (
        PiecewiseAggregateApproximation(
            n_segments=current_n_segments
        )
        .fit_transform(X_ts_p)[:, :, 0]
    )

    if current_breakpoint_mode == "empirical":
        breakpoints_p = empirical_breakpoints(
            paa_p.ravel(),
            current_alphabet_size,
        )
    else:
        breakpoints_p = sax_breakpoints(
            current_alphabet_size
        )

    codes_p = discretize_paa(
        paa_p,
        breakpoints_p,
    )

    symbol_dist_p = sax_symbol_distance_table(
        current_alphabet_size,
        breakpoints_p,
    )

    # -------------------------------------------------------------------------
    # Pairwise SAX MINDIST and Euclidean tie-breaking
    # -------------------------------------------------------------------------

    terms_p = panel_norm_p.columns.to_list()

    mindist_p = sax_mindist_matrix(
        sax_codes=codes_p[:, :, None],
        terms=terms_p,
        n_timestamps=panel_norm_p.shape[0],
        n_segments=current_n_segments,
        alphabet_size=current_alphabet_size,
        symbol_dist=symbol_dist_p,
    )

    euclid_p = true_euclidean_distance_matrix(
        panel_sax_p
    )

    mindist_tiebroken_p = break_mindist_ties(
        mindist_p,
        euclid_p,
        eps=current_tie_eps,
    )

    # -------------------------------------------------------------------------
    # Cluster at the fixed baseline k
    # -------------------------------------------------------------------------

    raw_labels = hierarchical_labels(
        mindist_tiebroken_p,
        k=BASELINE_K,
        method=current_linkage_method,
    )

    labels_p = pd.Series(
        raw_labels,
        index=mindist_tiebroken_p.index,
        name="cluster",
    ).astype(int)

    if labels_p.index.has_duplicates:
        raise ValueError(
            "The perturbed clustering contains duplicate term indices."
        )

    return labels_p


def ari_on_intersection(
    baseline_labels: pd.Series,
    perturbed_labels: pd.Series,
) -> tuple[float, int]:
    """
    Compare two cluster partitions using only terms present in both runs.

    Filtering perturbations may change the retained term set, so the labels are
    aligned by their shared term indices before calculating ARI.
    """
    if not isinstance(baseline_labels, pd.Series):
        raise TypeError("baseline_labels must be a pandas Series.")

    if not isinstance(perturbed_labels, pd.Series):
        raise TypeError("perturbed_labels must be a pandas Series.")

    if baseline_labels.index.has_duplicates:
        raise ValueError(
            "baseline_labels contains duplicate term indices."
        )

    if perturbed_labels.index.has_duplicates:
        raise ValueError(
            "perturbed_labels contains duplicate term indices."
        )

    common_terms = (
        baseline_labels.index
        .intersection(perturbed_labels.index)
        .sort_values()
    )

    if len(common_terms) < 2:
        return float("nan"), len(common_terms)

    aligned = pd.DataFrame(
        {
            "baseline": baseline_labels.loc[common_terms],
            "perturbed": perturbed_labels.loc[common_terms],
        }
    ).dropna()

    n_common = len(aligned)

    if n_common < 2:
        return float("nan"), n_common

    ari = adjusted_rand_score(
        aligned["baseline"].to_numpy(),
        aligned["perturbed"].to_numpy(),
    )

    return float(ari), n_common

In [22]:
# -----------------------------------------------------------------------------
# OFAAT sweep execution
# -----------------------------------------------------------------------------

sensitivity_rows = []


# -----------------------------------------------------------------------------
# Helper for safe display and comparison of parameter values
# -----------------------------------------------------------------------------

def values_equal(value_a, value_b) -> bool:
    """
    Compare scalar configuration values safely.

    Handles strings, integers, floats, NumPy scalars, and NaN values.
    """
    if pd.isna(value_a) and pd.isna(value_b):
        return True

    if isinstance(value_a, (float, np.floating)) or isinstance(
        value_b,
        (float, np.floating),
    ):
        try:
            return bool(np.isclose(value_a, value_b))
        except (TypeError, ValueError):
            pass

    return value_a == value_b


def serialize_config_value(value):
    """
    Convert configuration values to CSV-friendly representations.
    """
    if isinstance(value, np.generic):
        return value.item()

    return value


# -----------------------------------------------------------------------------
# Positive control
#
# Evaluate the baseline system configuration with zero changes. This verifies
# that the proxy pipeline reproduces the finalized notebook clustering.
# -----------------------------------------------------------------------------

control_labels = run_pipeline_proxy(rerun_from="filtering")

control_ari, control_n = ari_on_intersection(
    BASELINE_LABELS,
    control_labels,
)

sensitivity_rows.append(
    {
        "Pipeline Stage": "Control",
        "Parameter": "NO_CHANGE (Proxy Control)",
        "Baseline Value": "N/A",
        "Perturbed Value": "N/A",
        "Direction": "Control",
        "Surviving N": control_n,
        "Surviving Share": (
            control_n / len(BASELINE_LABELS)
            if len(BASELINE_LABELS) > 0
            else np.nan
        ),
        "ARI vs Baseline": control_ari,
        "Run Status": "success",
        "Error": "",
    }
)


# -----------------------------------------------------------------------------
# Run full OFAAT parameter sweep
# -----------------------------------------------------------------------------

for stage, parameters in PERTURBATIONS.items():
    rerun_tier = STAGE_RERUN_TIER[stage]

    for parameter, candidate_values in parameters.items():
        if parameter not in BASELINE_CONFIG:
            raise KeyError(
                f"{parameter!r} is included in PERTURBATIONS but is missing "
                "from BASELINE_CONFIG."
            )

        baseline_value = BASELINE_CONFIG[parameter]

        # Permit either a list/tuple or an accidentally supplied scalar.
        if not isinstance(
            candidate_values,
            (list, tuple, set, np.ndarray, pd.Series),
        ):
            candidate_values = [candidate_values]

        for perturbed_value in candidate_values:
            # Do not rerun the exact baseline configuration.
            if values_equal(perturbed_value, baseline_value):
                continue

            if (
                isinstance(perturbed_value, (int, float, np.number))
                and isinstance(baseline_value, (int, float, np.number))
            ):
                if perturbed_value < baseline_value:
                    direction = "Lower"
                elif perturbed_value > baseline_value:
                    direction = "Higher"
                else:
                    direction = "Alternative"
            else:
                direction = "Alternative"

            try:
                with isolated_global_override(
                    parameter,
                    perturbed_value,
                ):
                    perturbed_labels = run_pipeline_proxy(
                        rerun_from=rerun_tier
                    )

                ari, n_common = ari_on_intersection(
                    BASELINE_LABELS,
                    perturbed_labels,
                )

                sensitivity_rows.append(
                    {
                        "Pipeline Stage": stage,
                        "Parameter": parameter,
                        "Baseline Value": serialize_config_value(
                            baseline_value
                        ),
                        "Perturbed Value": serialize_config_value(
                            perturbed_value
                        ),
                        "Direction": direction,
                        "Surviving N": n_common,
                        "Surviving Share": (
                            n_common / len(BASELINE_LABELS)
                            if len(BASELINE_LABELS) > 0
                            else np.nan
                        ),
                        "ARI vs Baseline": ari,
                        "Run Status": "success",
                        "Error": "",
                    }
                )

            except Exception as exc:
                sensitivity_rows.append(
                    {
                        "Pipeline Stage": stage,
                        "Parameter": parameter,
                        "Baseline Value": serialize_config_value(
                            baseline_value
                        ),
                        "Perturbed Value": serialize_config_value(
                            perturbed_value
                        ),
                        "Direction": direction,
                        "Surviving N": np.nan,
                        "Surviving Share": np.nan,
                        "ARI vs Baseline": np.nan,
                        "Run Status": "failed",
                        "Error": f"{type(exc).__name__}: {exc}",
                    }
                )

                print(
                    f"WARNING: OFAAT run failed for "
                    f"{parameter}={perturbed_value!r}: {exc}"
                )


# -----------------------------------------------------------------------------
# Post-processing
# -----------------------------------------------------------------------------

sensitivity_df = pd.DataFrame(sensitivity_rows)

stage_order = [
    "Control",
    "Preprocessing",
    "Filtering",
    "Representation",
]

sensitivity_df["Pipeline Stage"] = pd.Categorical(
    sensitivity_df["Pipeline Stage"],
    categories=stage_order,
    ordered=True,
)

sensitivity_df = (
    sensitivity_df
    .sort_values(
        [
            "Pipeline Stage",
            "Parameter",
            "Perturbed Value",
        ],
        key=lambda col: col.astype(str)
        if col.name == "Perturbed Value"
        else col,
    )
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# Add within-parameter robustness summaries
# -----------------------------------------------------------------------------

successful_runs = sensitivity_df[
    (sensitivity_df["Run Status"] == "success")
    & (sensitivity_df["Pipeline Stage"] != "Control")
].copy()

parameter_summary = (
    successful_runs
    .groupby(
        ["Pipeline Stage", "Parameter"],
        observed=True,
        as_index=False,
    )
    .agg(
        n_perturbations=("ARI vs Baseline", "size"),
        min_ari=("ARI vs Baseline", "min"),
        median_ari=("ARI vs Baseline", "median"),
        mean_ari=("ARI vs Baseline", "mean"),
        max_ari=("ARI vs Baseline", "max"),
        min_surviving_n=("Surviving N", "min"),
        mean_surviving_share=("Surviving Share", "mean"),
    )
)

parameter_summary["ari_range"] = (
    parameter_summary["max_ari"]
    - parameter_summary["min_ari"]
)

parameter_summary = parameter_summary.sort_values(
    ["min_ari", "median_ari"],
    ascending=[True, True],
).reset_index(drop=True)


# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------

ROBUSTNESS_DIR.mkdir(parents=True, exist_ok=True)

sensitivity_output_path = (
    ROBUSTNESS_DIR / "ofaat_sensitivity_analysis.csv"
)

summary_output_path = (
    ROBUSTNESS_DIR / "ofaat_parameter_summary.csv"
)

sensitivity_df.to_csv(
    sensitivity_output_path,
    index=False,
)

parameter_summary.to_csv(
    summary_output_path,
    index=False,
)


# -----------------------------------------------------------------------------
# Guard-rail checks
# -----------------------------------------------------------------------------

control_rows = sensitivity_df[
    sensitivity_df["Pipeline Stage"] == "Control"
]

if len(control_rows) != 1:
    raise RuntimeError(
        f"Expected exactly one OFAAT control row; found "
        f"{len(control_rows)}."
    )

control_ari_value = control_rows.iloc[0]["ARI vs Baseline"]

if pd.isna(control_ari_value):
    print(
        "CRITICAL WARNING: The control ARI could not be calculated."
    )
elif control_ari_value < 0.95:
    print(
        f"CRITICAL WARNING: Control-row ARI is low "
        f"({control_ari_value:.3f})."
    )
    print(
        "The proxy pipeline does not reproduce the notebook baseline. "
        "Interpret the perturbation results only after resolving this mismatch."
    )
else:
    print(
        f"Proxy control passed: ARI = {control_ari_value:.3f}."
    )


failed_runs = sensitivity_df[
    sensitivity_df["Run Status"] == "failed"
]

if len(failed_runs) > 0:
    print(
        f"\n{len(failed_runs)} OFAAT runs failed:"
    )
    print(
        failed_runs[
            [
                "Pipeline Stage",
                "Parameter",
                "Perturbed Value",
                "Error",
            ]
        ].to_string(index=False)
    )


low_robustness = sensitivity_df[
    (sensitivity_df["Run Status"] == "success")
    & (sensitivity_df["Pipeline Stage"] != "Control")
    & (sensitivity_df["ARI vs Baseline"] < 0.75)
]

if len(low_robustness) > 0:
    print(
        "\nPerturbations below the ARI 0.75 robustness threshold:"
    )
    print(
        low_robustness[
            [
                "Pipeline Stage",
                "Parameter",
                "Baseline Value",
                "Perturbed Value",
                "Surviving N",
                "Surviving Share",
                "ARI vs Baseline",
            ]
        ].to_string(index=False)
    )
else:
    print(
        "\nAll successful perturbations scored at least 0.75 ARI "
        "against the baseline partition."
    )


# -----------------------------------------------------------------------------
# Identify the most sensitive parameters
# -----------------------------------------------------------------------------

if not parameter_summary.empty:
    print("\nParameter-level robustness summary:")
    print(
        parameter_summary.to_string(
            index=False,
            float_format=lambda value: f"{value:.3f}",
        )
    )

    most_sensitive = parameter_summary.iloc[0]

    print(
        "\nMost sensitive parameter based on minimum ARI: "
        f"{most_sensitive['Parameter']} "
        f"(minimum ARI = {most_sensitive['min_ari']:.3f})."
    )


# -----------------------------------------------------------------------------
# Styled display
# -----------------------------------------------------------------------------

styled_sensitivity = (
    sensitivity_df.style
    .background_gradient(
        subset=["ARI vs Baseline"],
        cmap="RdYlGn",
        vmin=0.0,
        vmax=1.0,
    )
    .format(
        {
            "ARI vs Baseline": "{:.3f}",
            "Surviving Share": "{:.1%}",
        },
        na_rep="—",
    )
    .set_caption(
        f"OFAAT sensitivity analysis versus the fixed baseline partition "
        f"(N={len(BASELINE_LABELS)}, K={BASELINE_K}, "
        f"linkage={BASELINE_CONFIG['LINKAGE_METHOD']}). "
        "Each row changes one parameter only; the control row should "
        "approach ARI = 1.000."
    )
    .hide(axis="index")
)

styled_sensitivity

Proxy control passed: ARI = 1.000.

Perturbations below the ARI 0.75 robustness threshold:
Pipeline Stage         Parameter Baseline Value Perturbed Value  Surviving N  Surviving Share  ARI vs Baseline
 Preprocessing DENOISE_POLYORDER              2               3          847         1.000000         0.483466
 Preprocessing    DENOISE_WINDOW              9              15          847         1.000000         0.319916
 Preprocessing    DENOISE_WINDOW              9              21          847         1.000000         0.319302
 Preprocessing    DENOISE_WINDOW              9               7          847         1.000000         0.393338
 Preprocessing    DETREND_WINDOW             91              31          847         1.000000         0.290878
 Preprocessing    DETREND_WINDOW             91              61          847         1.000000         0.350863
     Filtering    MAX_ZERO_SHARE            0.5             0.3          762         0.899646         0.468648
     Filtering    MAX

Pipeline Stage,Parameter,Baseline Value,Perturbed Value,Direction,Surviving N,Surviving Share,ARI vs Baseline,Run Status,Error
Control,NO_CHANGE (Proxy Control),N/A,N/A,Control,847,100.0%,1.000,success,
Preprocessing,DENOISE_POLYORDER,2,3,Higher,847,100.0%,0.483,success,
Preprocessing,DENOISE_WINDOW,9,15,Higher,847,100.0%,0.320,success,
Preprocessing,DENOISE_WINDOW,9,21,Higher,847,100.0%,0.319,success,
Preprocessing,DENOISE_WINDOW,9,7,Lower,847,100.0%,0.393,success,
Preprocessing,DETREND_WINDOW,91,31,Lower,847,100.0%,0.291,success,
Preprocessing,DETREND_WINDOW,91,61,Lower,847,100.0%,0.351,success,
Filtering,MAX_INTERPOLATE_GAP,7,14,Higher,847,100.0%,1.000,success,
Filtering,MAX_INTERPOLATE_GAP,7,21,Higher,847,100.0%,1.000,success,
Filtering,MAX_MISSING_SHARE,0.020000,0.050000,Higher,847,100.0%,1.000,success,


## 7. Final visualizations

This section uses the fixed Step 4 `labels_final` and the validated cluster profiles to create cluster archetype panels, shape dictionaries, and delay-embedding phase portraits. It does not refit or alter the clustering.


In [23]:
# -----------------------------------------------------------------------------
# Shared periodicity diagnostic
# Distinguishes "genuinely periodic" from "periodogram picked a peak out of an
# otherwise flat/noisy spectrum" -- the bug behind clusters 1, 6, 9 collapsing
# to a ~2-week display window, and the reason clusters 1/2/5/7/10 show comet
# shapes rather than loops in the phase portraits (which is the CORRECT
# behavior for event-driven series, but needs to be labeled as such rather
# than silently treated the same as real periodicity).
# -----------------------------------------------------------------------------
from scipy.signal import periodogram

def periodicity_diagnostics(
    series: np.ndarray,
    min_period: int = 3,
    max_period: int = 730,
    significance_ratio: float = 3.0,
) -> tuple[int, bool, float]:
    """
    Returns (dominant_period, is_significant, concentration).

    concentration = power at the dominant period / mean power across the
    searched band. is_significant = concentration >= significance_ratio,
    i.e. the peak clearly stands out from a roughly flat (noise-like)
    spectrum rather than being an arbitrary highest point in the noise floor.
    """
    x = series - np.nanmean(series)
    freqs, power = periodogram(x, fs=1.0)
    periods = np.zeros_like(freqs)
    valid = freqs > 0
    periods[valid] = 1.0 / freqs[valid]
    mask = (periods >= min_period) & (periods <= max_period)

    if not mask.any() or power[mask].sum() == 0:
        return min_period, False, 0.0

    band_power = power[mask]
    band_periods = periods[mask]
    peak_idx = np.argmax(band_power)
    peak_period = int(round(band_periods[peak_idx]))
    concentration = float(band_power[peak_idx] / (band_power.mean() + 1e-12))
    is_significant = concentration >= significance_ratio

    return peak_period, is_significant, concentration


def dominant_period(series: np.ndarray, min_period: int = 3, max_period: int = 730) -> int:
    """Back-compat thin wrapper (period only, no significance check)."""
    period, _, _ = periodicity_diagnostics(series, min_period, max_period)
    return period


def auto_smooth_window(series: np.ndarray, max_lag: int = 60) -> int:
    """
    Smoothing window from the series' own ACF decay: the lag where
    autocorrelation first drops inside the +/-1.96/sqrt(n) noise band.
    This is deliberately a SEPARATE quantity from the display window below --
    conflating the two in a single 'window=Xd' label was the source of the
    confusing archetype titles.
    """
    x = pd.Series(series).interpolate().bfill().ffill().to_numpy()
    n = len(x)
    a = acf(x, nlags=min(max_lag, n // 3), fft=True)
    threshold = 1.96 / np.sqrt(n)
    below = np.where(np.abs(a[1:]) < threshold)[0]
    window = int(below[0] + 1) if len(below) else 7
    return max(3, window | 1)


from statsmodels.tsa.stattools import acf


def select_display_window(
    panel: pd.DataFrame,
    terms: list[str],
    cycles: float = 2.5,
    significance_ratio: float = 3.0,
    min_significant_share: float = 0.5,
) -> tuple[str, str, str]:
    """
    Show `cycles` periods of the cluster's dominant period IF a majority of
    its member terms actually have a significant periodic peak. Otherwise,
    fall back to the FULL available range -- shrinking the window based on a
    meaningless periodogram peak (as happened for clusters 1, 6, 9) actively
    hides the true pattern instead of revealing it.

    Returns (start, end, basis) where `basis` is a short string documenting
    which branch was taken, for direct inclusion in the plot title.
    """
    diags = [
        periodicity_diagnostics(panel[t].to_numpy(dtype=float), significance_ratio=significance_ratio)
        for t in terms if t in panel.columns
    ]
    end = panel.index.max()

    if not diags:
        return panel.index.min().strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"), "full range (no terms)"

    sig_flags = [d[1] for d in diags]
    frac_significant = float(np.mean(sig_flags))

    if frac_significant >= min_significant_share:
        sig_periods = [d[0] for d in diags if d[1]]
        period = int(np.median(sig_periods))
        start = max(end - pd.Timedelta(days=int(period * cycles)), panel.index.min())
        basis = f"periodic, period={period}d ({frac_significant:.0%} of terms significant)"
    else:
        start = panel.index.min()
        basis = f"full range (only {frac_significant:.0%} of terms show significant periodicity)"

    return start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"), basis

# -----------------------------------------------------------------------------
# Cluster archetype panel: distinct-shape small multiples
# One subplot per cluster: centroid trajectory + IQR band, each on its own
# auto-selected window/smoothing, so genuinely different dynamic ranges and
# cycle lengths stay visible instead of being normalized away.
# -----------------------------------------------------------------------------

ARCHETYPE_DIR = OUTPUT_DIR / "07_visualization" / "cluster_archetypes"
ARCHETYPE_DIR.mkdir(parents=True, exist_ok=True)

clusters_sorted = sorted(labels_final.unique())
ncols = 3
nrows = int(np.ceil(len(clusters_sorted) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.3 * nrows), squeeze=False)

archetype_meta = []
for i, cl in enumerate(clusters_sorted):
    ax = axes[i // ncols, i % ncols]
    members = labels_final[labels_final == cl].index.intersection(panel_norm.columns).to_list()

    display_start, display_end, basis = select_display_window(panel_norm, members, cycles=2.5)
    sub = panel_norm.loc[display_start:display_end, members]

    smooth_window = auto_smooth_window(sub.mean(axis=1).to_numpy())
    smoothed = sub.rolling(smooth_window, center=True, min_periods=1).mean()

    centroid = smoothed.mean(axis=1)
    q25, q75 = smoothed.quantile(0.25, axis=1), smoothed.quantile(0.75, axis=1)

    ax.fill_between(smoothed.index, q25, q75, alpha=0.25, linewidth=0)
    ax.plot(smoothed.index, centroid, linewidth=2.0)

    # Two separate, clearly-labeled quantities: which dates are shown (basis)
    # vs. how much the display trace is smoothed (smooth_window).
    ax.set_title(f"Cluster {cl}  (n={len(members)}, smooth={smooth_window}d)\n{basis}", fontsize=8.5)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    archetype_meta.append({
        "cluster": int(cl), "n_terms": len(members),
        "display_start": display_start, "display_end": display_end,
        "display_window_basis": basis,
        "smooth_window_days": smooth_window,
    })

for j in range(len(clusters_sorted), nrows * ncols):
    axes[j // ncols, j % ncols].axis("off")

fig.suptitle("Cluster archetypes: centroid \u00b1 IQR (display window and smoothing window are independent)", fontsize=12.5)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(ARCHETYPE_DIR / "cluster_archetype_panel.png", dpi=300, bbox_inches="tight")
plt.show()

pd.DataFrame(archetype_meta).to_csv(ARCHETYPE_DIR / "cluster_archetype_window_selection.csv", index=False)

In [24]:
from statsmodels.tsa.seasonal import STL

def shape_signature(series: pd.Series, period: int) -> dict:
    """
    A small feature vector describing a series' shape -- trend strength,
    seasonal strength, spikiness, dominant period -- independent of SAX
    strings, so new terms can be compared to existing cluster archetypes
    directly, without rebuilding the MINDIST matrix.
    """
    x = series.interpolate().bfill().ffill()
    try:
        res = STL(x, period=max(3, period), robust=True).fit()
        var_resid = np.var(res.resid)
        trend_strength = max(0.0, 1 - var_resid / np.var(res.trend + res.resid))
        seasonal_strength = max(0.0, 1 - var_resid / np.var(res.seasonal + res.resid))
    except Exception:
        trend_strength, seasonal_strength = np.nan, np.nan

    return {
        "trend_strength": trend_strength,
        "seasonal_strength": seasonal_strength,
        "spikiness_kurtosis": float(pd.Series(x).kurt()),
        "dominant_period_days": dominant_period(x.to_numpy()),
    }


cluster_dictionary = []
for cl in clusters_sorted:
    members = labels_final[labels_final == cl].index.intersection(panel_norm.columns).to_list()
    centroid = panel_norm[members].mean(axis=1)
    sig = shape_signature(centroid, period=dominant_period(centroid.to_numpy()))
    sig.update({"cluster": int(cl), "n_terms": len(members)})
    cluster_dictionary.append(sig)

cluster_dictionary_df = pd.DataFrame(cluster_dictionary).set_index("cluster")
cluster_dictionary_df.to_csv(OUTPUT_DIR / "07_visualization" / "cluster_shape_dictionary.csv")


def classify_new_term(new_series: pd.Series, dictionary: pd.DataFrame) -> pd.Series:
    """Rank existing cluster archetypes by similarity to a brand-new term's shape."""
    cols = ["trend_strength", "seasonal_strength", "spikiness_kurtosis", "dominant_period_days"]
    sig = pd.Series(shape_signature(new_series, dominant_period(new_series.to_numpy())))[cols]
    ref = dictionary[cols]
    ref_std = (ref - ref.mean()) / ref.std().replace(0, 1)
    sig_std = (sig - ref.mean()) / ref.std().replace(0, 1)
    return np.sqrt(((ref_std - sig_std) ** 2).sum(axis=1)).sort_values()

In [25]:
# -----------------------------------------------------------------------------
# Delay-embedding phase-portrait panel
# One subplot per cluster: x(t) vs x(t - lag), lag = the cluster's own
# dominant period (from periodicity_diagnostics, defined above). A closed
# loop indicates a periodic shape; a diagonal streak indicates a trending
# shape; a diffuse "comet" (dense clump + a few far-flung points) indicates
# event-driven behavior rather than a real cycle -- flagged explicitly below
# rather than left ambiguous, since a comet is the CORRECT shape for a
# one-off-event cluster, not a failed periodic one.
# -----------------------------------------------------------------------------
PHASE_DIR = OUTPUT_DIR / "07_visualization" / "phase_portraits"
PHASE_DIR.mkdir(parents=True, exist_ok=True)


def phase_embed(x: np.ndarray, lag: int) -> tuple[np.ndarray, np.ndarray]:
    """Standard delay embedding: pairs (x[t], x[t - lag]) for t >= lag."""
    if lag < 1 or lag >= len(x):
        lag = max(1, len(x) // 4)
    return x[lag:], x[:-lag]


clusters_sorted = sorted(labels_final.unique())
ncols = 3
nrows = int(np.ceil(len(clusters_sorted) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.4 * nrows), squeeze=False)

phase_meta = []
for i, cl in enumerate(clusters_sorted):
    ax = axes[i // ncols, i % ncols]
    members = labels_final[labels_final == cl].index.intersection(panel_norm.columns).to_list()

    centroid = panel_norm[members].mean(axis=1).interpolate().bfill().ffill().to_numpy()
    lag, is_significant, concentration = periodicity_diagnostics(centroid)

    x_t, x_lag = phase_embed(centroid, lag)
    t_idx = np.arange(len(x_t))
    ax.scatter(x_lag, x_t, c=t_idx, cmap="viridis", s=6, alpha=0.7, linewidths=0)
    ax.plot(x_lag, x_t, color="gray", alpha=0.15, linewidth=0.6)

    tag = "periodic" if is_significant else "event-driven, not periodic"
    ax.set_title(f"Cluster {cl}  (n={len(members)}, lag={lag}d)\n{tag}  [concentration={concentration:.1f}]", fontsize=8.5)
    ax.set_xlabel(f"x(t - {lag})", fontsize=8)
    ax.set_ylabel("x(t)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_aspect("equal", adjustable="box")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    phase_meta.append({
        "cluster": int(cl), "n_terms": len(members),
        "embedding_lag_days": lag,
        "is_significant_periodicity": is_significant,
        "spectral_concentration": concentration,
    })

for j in range(len(clusters_sorted), nrows * ncols):
    axes[j // ncols, j % ncols].axis("off")

fig.suptitle("Cluster phase portraits: delay embedding, with periodicity significance flagged", fontsize=12.5)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PHASE_DIR / "cluster_phase_portrait_panel.png", dpi=300, bbox_inches="tight")
plt.show()

pd.DataFrame(phase_meta).to_csv(PHASE_DIR / "phase_portrait_lag_selection.csv", index=False)